# Модель прогнозирования финансовых цен

Модель прогнозирования коридора цен на золото на базе новостной ленты канала Telegram [MarketTwits](https://t.me/markettwits) и показателей технических индикаторов. В ходе работы обучается три модели, прогнозирующие за установленный период:
- максимальную цену;
- минимальную цену;
- рост/падение цены (относительно цены закрытия).

## Анализ требований

Итоговые модели должны выдавать прогнозируемые значения изменения цены финансового актива - границы коридора и направление движения относительно цены закрытия за период отработки новости/финансового отчета. Данные модели планируются для использования в качестве источника сигналов рекомендательной торговой системы, а в качестве развития проекта и для автоматической торговой системы.  
Так как абсолютные значения цены актива могут лежать вне анализируемых диапазонов цен (обновление максимума/минимума), то анализу подвергается *относительная* разница между ценой закрытия и прогнозируемыми значениями. В рамках одного объекта это означает, что после получения значений всех технических индикаторов, выбранных для анализа, из тех значений, которые отражаются в масштабе цены актива (например, средние за период, Облако Ишимоку и т.п.), и прогнозируемых значений будет вычтена цена закрытия. Цена закрытия выбрана в качестве опорного значения, так как она является наиболее приближенной к потенциальной цене открытия сделки в текущем моменте (в условиях низкой волатильности).  
В качестве источника текущих котировок финансового актива выступает брокер [Финам](https://www.finam.ru/), источником новостей - Telegram-канал [MarketTwits](https://t.me/markettwits).

Исходные данные для обучения модели:
- исторические данные по золоту (GOLD), доступны на странице [Актуальных архивов исторических данных (котировок)](https://daytradingschool.ru/trejderu/skachat-kotirovki-istoricheskie-dannye-po-fyuchersam/);
- новостная лента канала MarketTwits.

### Формирование аналитической задачи

На данном этапе задачу проекта сводим к решению задачи регрессии для моделей получения значений максимальной и минимальной цен и классификации для модели определения направления движения цены.  
В качестве базовых типов моделей отдаем предпочтение классическим против моделей временных рядов, так как финансовые ряды часто нестационарны, а модели временных рядов требуют проверки на стационарность и дифференцирования, что может искажать экономическую интерпретацию, а также модели временных рядов в базовой форме слабо интегрируют внешние переменные, в то время как классические модели потенциально могут включать больше признаков.  

Так как в алгоритмах решения задачи регрессии не учитываются исторические данные, то необходима генерация признаков, обеспечивающих ретроспективность. Для этого предлагается использовать значения технических индикаторов и предыдущих записей.  

При прогнозировании направления движения цены актива и моделировании принятия решения на открытие позиции действия трейдера более сходны с последовательным анализом отобранной информации, таким образом, в качестве алгоритма обучения наиболее целесообразно использование **дерева решений** и его модификаций. Для оценки качества моделей оптимально использовать среднюю абсолютную ошибку в процентах **MAPE** и коэффициент детерминации $R^2$, так как итоговое рассчетное значение MAPE можно использовать в качестве одного из рассчетных параметров для формирования политики выставления страховочных заявок (stop-loss). В моделе определения направления движения цены наиболее важно сведение к минимуму ложных срабатываний, соответственно для их оценки выбираем точность предсказаний (PPV, precision) (лучше пропустить несколько удачных точек входа, чем отработать неудачные).

Также, исходя из последнего замечания, уровень значимости $\alpha$ выбираем 0.05.

### Выбор общих параметров моделей, определяемых спецификой задачи

**Продолжительность сделки**  
Разрабатываемая стратегия предполагает совершение сделок в рамках дейтрейдинга (Day Trading). Если говорить о среднем по рынку, то сделки в дейтрейдинге чаще всего занимают от 1 до 2 часов. Это оптимальное время для трейдеров, которые ищут возможности на основе внутридневных колебаний, но не хотят держать позиции слишком долго, чтобы избежать рисков ночных гэпов или изменения рыночной ситуации.  
Наиболее сильное движение рынка в рамках дня обычно укладывается в первые 30–90 минут после выхода важной новости или открытия основной торговой сессии.  
В первичной версии выбираем значение в **1 час (60 минут)**.
Данное значение будет использовано для получения целевых значений обучающей выборки.

**Период совершения сделок**  
Торговля целевым активом (фьючерс на золото) на Московской бирже проводится в несколько сессий: 
- Аукцион открытия 08:50 - 09:00
- Утренняя сессия 09:00 - 10:00
- Основная сессия 10:00 - 14:00
- Клиринг 14:00 - 14:05
- Основная сессия 14:05 - 18:50
- Клиринг 18:50 - 19:05
- Вечерняя сессия 19:05 - 23:50

Совершение сделок целесообразно проводить в период основных торговых сессий, так как в остальные периоды низкая ликвидность может стать причиной повышенного риска и роста волатильности. Также исходя из выбранной продолжительности сделки в 1 час, ограничиваем время для отрытия позиций следующим периодом: *10:00 - 17:30*.

Так как исходные данные представляют из себя склейку торговых данных по фьючерсам, то будет необходимо на этапе очистки удалить данные в районе дней экспирации фьючерсов (2 последних дня торговли фьючерсом и 1-2 последующих дней в зависимости от величины периодов технических индикаторов для исключения участия свечей дней экспирации в вычислении их значений).

При техническом анализе японских свечей выделяемые патерны включают в себя обычно от двух до пяти свечей. Считается, что в дейтрейдинге лучше работают следующие:
- разворотные паттерны:
    - Поглощение (Bullish/Bearish Engulfing);
    - Молот и Повешенный (Hammer/Hanging Man);
    - Просвет в облаках и Завеса темных туч (Piercing Line/Dark Cloud Cover);
    - Дожи (Doji, Dragonfly Doji, Gravestone Doji);
- паттерны продолжения тренда:
    - Три белых солдата / Три черных вороны;
    - Методы трех (Rising/Falling Three Methods);
- моментумные паттерны:
    - Большая белая (черная) свеча;
    - Гэп (разрыв цены).
    
Для анализа свечных патернов наиболее оптимальны 15-минутные свечи, вышеуказанные патерны состоят максимум из трех свечей. Таким образом для учета ретроспективы к каждому объекту (минутной свече) предлагается присоединять значения цен (открытия/закрытия/максимальной/минимальной) трех предыдущих завершенных 15-минутных свечей, соответственно для генерации значений индикаторов технического анализа необходимо использовать 15-минутные свечи.

Внутридневная торговля (дейтрейдинг) не имеет прямой зависимости между днем недели и направлением сделок. Однако есть определенные особенности, которые могут влиять на торговую активность в разные дни:
- понедельник часто характеризуется повышенной волатильностью, так как трейдеры возвращаются к торгам после выходных и начинают активно открывать позиции;
- вторник-четверг обычно являются наиболее стабильными днями для внутридневной торговли, когда рынок работает в обычном режиме;
- пятница может быть менее предсказуемой из-за:
    - закрытия позиций перед выходными;
    - повышенной активности на новостях;
    - снижения ликвидности ближе к концу дня.  
    
В связи с этим имеет смысл включение дня недели в качестве одного из факторов.

### Выбор технических индикаторов
Оптимальными индикаторами для дейтрейдинга (30–90 минут) являются:
1. Скользящие средние (Moving Averages, MA)
    - EMA 9 и EMA 21 - Для краткосрочного тренда и входов.
    - SMA 50 и SMA 200  Для глобального тренда на M15/M30.  
Используются для определения направления тренда и поиска зон поддержки/сопротивления.  
Стратегия "Пересечение EMA 9 и EMA 21" дает сигналы на покупку/продажу.  
Работает хорошо на M15/M30.

2. Индикатор объема (Volume, OBV, VSA)
    - VSA (Volume Spread Analysis) помогает оценить намерения крупных игроков.
    - OBV (On Balance Volume) помогает определить направление тренда на основе объема.  
Полезен для подтверждения ложных пробоев и определения силы тренда.  
Работает на M5/M15.

3. Индекс относительной силы (RSI) Период 14 или 7
    - Выше 70 - рынок перекуплен - возможен разворот вниз.
    - Ниже 30 - рынок перепродан - возможен разворот вверх.  
Полезен для поиска входов на откатах.  
Работает на M15/M30.

4. Стохастик (Stochastic Oscillator) Период 14, 3, 3 (или 8, 3, 3 для агрессивного входа)  
Показывает зоны перекупленности и перепроданности, но лучше работает во флете.  
Хорошо работает в комбинации с RSI.  
Работает на M5/M15.

5. VWAP (Volume Weighted Average Price)  
Отображает среднюю цену с учетом объема, что помогает находить зоны поддержки/сопротивления.  
Хорошо работает для интрадейного трендового движения.  
Подходит для M5/M15.

6. Bollinger Bands (BB) Период 20  
Когда цена выходит за границы Bollinger Bands, возможен откат.  
Помогает находить моменты высокой волатильности.  
Работает на M15/M30.

7. MACD (Moving Average Convergence Divergence) Параметры: стандартные (12, 26, 9)  
Бычий сигнал: MACD пересекает сигнальную линию вверх.  
Медвежий сигнал: MACD пересекает сигнальную линию вниз.  
Подходит для фильтрации ложных сигналов.  
Работает на M15/M30.

Классическими комбинациями индикаторов для дейтрейдинга (30–90 минут) приняты:  
1. Трендовая стратегия
- EMA 9 и EMA 21 – для подтверждения тренда.
- VWAP – для определения зон входа.
- RSI – для оценки силы тренда.
- MACD – для подтверждения входа.
- Рабочие таймфреймы: M15/M30.
2. Контртрендовая стратегия (развороты)
- Bollinger Bands – для нахождения экстремумов.
- RSI + Стохастик – для поиска точек разворота.
- VSA – для оценки объема на экстремумах.
- Рабочие таймфреймы: M5/M15.
3. Пробойная стратегия
- SMA 50 + SMA 200 – определяют глобальный тренд.
- VWAP + Volume – подтверждают силу пробоя.
- RSI – подтверждает импульс.
- Рабочие таймфреймы: M15/M30.

На данной итерации выбираем технические индикаторы трендовой стратегии:
- EMA, период 9;
- EMA, период 21;
- VWAP, с начала торговой сессии;
- RSI, период 7.
- MACD, параметры (12, 26, 9).

In [1]:
# Глобальные параметры
alpha = 0.05 # параметр альфа
main_period = '15min' # период сжатия минутного графика свечей и рассчета технических индикаторов
main_period_int = 15
# Параметры технических индикаторов
period_EMA_fast = 9
period_EMA_slow = 21
period_RSI = 7
period_MACD_fast = 12
period_MACD_slow = 26
period_MACD_signal = 9
# Количество вычисляемых уровней расширения Фибоначчи
count_FE_levels = 7
# Значение для вычисления ближайших "круглых чисел"
round_price = 5
# Шаг цены инстумента
step_price = 0.1
# Период соранения позиции в минутах
period_holding_position = 60
# Количество свечей, обеспечивающих ретроспективу объектов
count_previous_candle = 3
# Количество знаков для округления значений технических индикаторов, 
# вычисляемых в масштабе, отличном от цены актива
round_ta = 2
# Порог для определения направления движения цены в процентах
level_price_movement_direction = 0.17

# Необходимость пересчета моделей
recalculating_model_up_down = True
recalculating_model_high = True
recalculating_model_low = True
# Поредел значения корреляции признаков для удаления
corr_limit = 0.7
# Фиксируем random_state
rs = 42

## Загрузка и обзор данных

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import mplfinance as fplt
import pandas_ta_classic as ta
import category_encoders as ce
import scipy.stats as stats
from plotly.subplots import make_subplots
# Регулярные выражения
import re
# Получение тональности финансовых текстов
import math
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

# Обучение модели  
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingClassifier, GradientBoostingRegressor
from xgboost import XGBClassifier, XGBRegressor
from sklearn import metrics # инструменты для оценки точности модели

# Подбор гиперпараметров моделей
from hyperopt import fmin, tpe, hp, space_eval, Trials, STATUS_OK

# Сериализация/десериализация
import joblib

d:\!Works\Projects\sf_pr7_financial_price_forecasting_model\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def plot_numerical_feature(data: pd.DataFrame, col: str, main_title: str):
    ''' Функция отрисовывает коробчатую диаграмму и гистограмму распределения
    и выводит статистические данные числового признака.
    Args:
        data (pd.DataFrame): датасет
        col (str): наименование признака
        main_title (str): расшифровка наименования признака для вывода
            в титуле графика
    Returns:
        None
    '''
    # Инициализация полотна на два графика с общей осью X
    fig_col = make_subplots(rows=2, cols=1,
                            shared_xaxes=True, # Общая ось Х
                            x_title=main_title,
                            y_title='Количество',
                            row_heights=[0.15, 0.85], # Относительная высота полотен
                            subplot_titles=(f"Распределение признака <br>\"{main_title}\"", ""),
                            vertical_spacing=0.02) # Высота разделителя
    # Построение коробчатой диаграммы
    fig_col.add_trace(go.Box(x=data[col],
                             marker_color = '#1729B0', # Цвет маркера
                             jitter=0.5, # Разнос точек выбросов по вертикали
                             name='',
                             showlegend=False),
                      row=1, col=1)
    # Построение гистограммы
    fig_col.add_trace(go.Histogram(x=data[col],
                                   marker_color = '#1729B0', # Цвет маркера
                                   name=main_title, # Название набора данных
                                   showlegend=False),
                      row=2, col=1)
    # Построение линии среднего значения
    fig_col.add_vline(x=data[col].mean(),
                      line=dict(color="Red", dash='dash')) # Стиль линии
    # Размер полотна
    fig_col.update_layout(autosize = True, bargap=0.1) # Расстояние между столбцами гистограммы
    # fig_col.show('png')
    fig_col.show()

    print('Статистические данные признака:')
    display(data[col].describe().round(2))

In [4]:
def plot_categorical_feature(data: pd.DataFrame, col: str, main_title: str):
    ''' Функция отрисовывает бары распределения и выводит статистические
    данные категориального признака. Количество баров ограничено 70, 
    длина подписи значений оси - 15 символов.
    Args:
        data (pd.DataFrame): датасет
        col (str): наименование признака
        main_title (str): расшифровка наименования признака для вывода
            в титуле графика
    Returns:
        None
    '''
    # Подготовка данных
    count_bar = 70
    len_xtext = 15
    df_tmp = data[col].value_counts().reset_index()
    df_tmp[col] = df_tmp[col].apply(lambda x: x[:len_xtext]+'...' if len(str(x)) > len_xtext else x)
    # Инициализация полотна
    fig_col = go.Figure(data=go.Bar(x=df_tmp[col].head(count_bar).index,
                                    y=df_tmp['count'].head(count_bar)))
    
    # Размер полотна
    fig_col.update_layout(autosize = True)
    # Заголовок диаграммы
    fig_col.update_layout(title=f"Распределение признака <br>\"{main_title}\"",
                          title_x=0.5, # Расположение посередине
                          xaxis_title=main_title, # Подпись Х
                          yaxis_title='Количество', # Подпись Y
                          xaxis=dict(tickvals=df_tmp[col].head(count_bar).index, # Оригинальные значения для оси X
                                     ticktext=df_tmp[col].head(count_bar))) # Новые подписи для оси X                   
    # fig_col.show('png')
    fig_col.show()
    
    print('Статистические данные признака:')
    display(data[col].describe())
    print('Top-10 значений (%):')
    display(data[col].value_counts(normalize=True).head(10).round(4)*100)

In [5]:
! pip freeze > requirements.txt

### Загрузка котировок финансового актива

In [6]:
df_init = pd.read_csv('./data/GOLD.txt')
df_init.info()
df_init.tail()

<class 'pandas.DataFrame'>
RangeIndex: 3010469 entries, 0 to 3010468
Data columns (total 9 columns):
 #   Column    Dtype  
---  ------    -----  
 0   <TICKER>  str    
 1   <PER>     int64  
 2   <DATE>    int64  
 3   <TIME>    int64  
 4   <OPEN>    float64
 5   <HIGH>    float64
 6   <LOW>     float64
 7   <CLOSE>   float64
 8   <VOL>     int64  
dtypes: float64(4), int64(4), str(1)
memory usage: 206.7 MB


,<TICKER>,<PER>,<DATE>,<TIME>,<OPEN>,<HIGH>,<LOW>,<CLOSE>,<VOL>
3010464,GOLD,1,20260731,234500,4071.8,4071.9,4070.5,4071.6,70
3010465,GOLD,1,20260731,234600,4070.8,4071.2,4070.7,4071.2,9
3010466,GOLD,1,20260731,234700,4071.2,4071.2,4071.0,4071.2,46
3010467,GOLD,1,20260731,234800,4071.0,4071.0,4070.1,4070.1,45
3010468,GOLD,1,20260731,234900,4070.9,4072.7,4070.0,4070.7,215


In [7]:
# Удаление неинформативных столбцов
df_init.drop(['<TICKER>', '<PER>'], axis=1, inplace=True)
# Переименование столбцов
df_init.rename(columns={'<OPEN>': 'open',
                        '<HIGH>': 'high',
                        '<LOW>': 'low',
                        '<CLOSE>': 'close',
                        '<VOL>': 'volume'}, 
               inplace=True)
# Преобразование времени свечей
df_init['dt'] = df_init['<DATE>'].astype(str) + ' ' + df_init['<TIME>'].astype(str)
df_init['dt'] = pd.to_datetime(df_init['dt'], format="%Y%m%d %H%M%S")
# Добавление дня недели
df_init['day_of_week'] = df_init['dt'].dt.day_name()
df_init = df_init[['dt', 'open', 'high', 'low', 'close', 'volume', 'day_of_week']]

df_M1 = df_init.copy()
df_M1.set_index('dt', inplace=True)

df_M1.info()
df_M1.tail()

<class 'pandas.DataFrame'>
DatetimeIndex: 3010469 entries, 2009-01-11 10:33:00 to 2026-07-31 23:49:00
Data columns (total 6 columns):
 #   Column       Dtype  
---  ------       -----  
 0   open         float64
 1   high         float64
 2   low          float64
 3   close        float64
 4   volume       int64  
 5   day_of_week  str    
dtypes: float64(4), int64(1), str(1)
memory usage: 160.8 MB


,open,high,low,close,volume,day_of_week
dt,,,,,,
2026-07-31 23:45:00,4071.8,4071.9,4070.5,4071.6,70,Friday
2026-07-31 23:46:00,4070.8,4071.2,4070.7,4071.2,9,Friday
2026-07-31 23:47:00,4071.2,4071.2,4071.0,4071.2,46,Friday
2026-07-31 23:48:00,4071.0,4071.0,4070.1,4070.1,45,Friday
2026-07-31 23:49:00,4070.9,4072.7,4070.0,4070.7,215,Friday


Датасет котировок на золото **df_M1** содержит значения минутных свечей и объема торгов начиная с 11 января 2009 года по 31 июля 2026 года, 3010469 строк.

Для обучения моделей возьмем период с 1 августа 2022 года по 31 июля 2026. Данный период необходим для поддержания баланса, так как если взять слишком много старых данных, модели могут зафиксировать устаревшие паттерны, а если выбрать только последний небольшой период, то существует риск переобучения на шуме. В условиях задачи решено опираться на 4 года данных. Так современная конъюнктура будет весомее, но при этом в выборке сохранится достаточная вариативность рыночных условий.

In [8]:
# Фильтрация записей за период
mask = (df_M1.index > '2022-08-01 00:00:00') & \
       (df_M1.index < '2026-08-01 00:00:00')
df_M1 = df_M1[mask]
df_M1.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 895517 entries, 2022-08-01 10:00:00 to 2026-07-31 23:49:00
Data columns (total 6 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   open         895517 non-null  float64
 1   high         895517 non-null  float64
 2   low          895517 non-null  float64
 3   close        895517 non-null  float64
 4   volume       895517 non-null  int64  
 5   day_of_week  895517 non-null  str    
dtypes: float64(4), int64(1), str(1)
memory usage: 47.8 MB


Датасет минутных свечей после отбора за период содержит 895517 записей.

Свернем полученный датасет до 15-минутного таймфрейма свечей.

In [9]:
# Сворачивание данных в 15-минутные свечи
df_main_period = df_M1.resample(main_period).agg({'open': 'first',   # Первая цена открытия за 15 минут
                                                  'high': 'max',     # Максимальная цена за 15 минут
                                                  'low': 'min',      # Минимальная цена за 15 минут
                                                  'close': 'last',   # Последняя цена закрытия за 15 минут
                                                  'volume': 'sum'})  # Сумма объема за 15 минут
df_main_period.dropna(axis=0, inplace=True)
print(df_main_period)

                       open    high     low   close  volume
dt                                                         
2022-08-01 10:00:00  1770.4  1777.0  1768.4  1774.6    4640
2022-08-01 10:15:00  1774.5  1775.9  1770.3  1771.6    2441
2022-08-01 10:30:00  1771.6  1771.8  1762.6  1767.3    5383
2022-08-01 10:45:00  1767.3  1769.7  1766.5  1769.5     616
2022-08-01 11:00:00  1769.2  1769.5  1765.6  1766.6    1441
...                     ...     ...     ...     ...     ...
2026-07-31 22:45:00  4074.7  4076.0  4072.0  4073.0    1394
2026-07-31 23:00:00  4073.0  4073.0  4067.1  4067.8    1218
2026-07-31 23:15:00  4067.9  4068.7  4066.6  4067.6     673
2026-07-31 23:30:00  4067.6  4073.0  4067.2  4071.8    1006
2026-07-31 23:45:00  4071.8  4072.7  4070.0  4070.7     385

[63539 rows x 5 columns]


Датасет котировок на золото **df_main_period** содержит значения 15-минутных свечей и объема торгов за ранее выбранный период и включает 63539 строк.

### Загрузка и обзор сообщений новостного канала

Произведем загрузку датасета, полученного при выполнении файла MsgDownload.ipynb.

In [10]:
df_msgs = pd.read_csv('./data/tg_messages.csv')
df_msgs.info()
df_msgs.head()

<class 'pandas.DataFrame'>
RangeIndex: 171856 entries, 0 to 171855
Data columns (total 2 columns):
 #   Column    Non-Null Count   Dtype
---  ------    --------------   -----
 0   date_utc  171856 non-null  str  
 1   message   171856 non-null  str  
dtypes: str(2)
memory usage: 2.6 MB


,date_utc,message
0,2026-07-31T21:01:39+00:00,❗️⚠️🇮🇷#иран #геополитика \nСША И ИЗРАИЛЬ ГОТОВ...
1,2026-07-31T19:06:20+00:00,"🇮🇷#иран #today #трамп \nТрамп заявил, что Уитк..."
2,2026-07-31T18:25:15+00:00,🚫✴️🇷🇺#майнинг #россия #крипто\nПравительство Р...
3,2026-07-31T18:22:45+00:00,"🇺🇸🇷🇺#санкции #россия \nЗаконопроект об ""адских..."
4,2026-07-31T18:04:08+00:00,❗️🇯🇵 #fx #япония #интервенции\nСША и Япония пр...


Приведем столбец date_utc ко времени с учетом часового пояса и проведем контрольный отбор за период.

In [11]:
df_msgs['dt'] = pd.to_datetime(df_msgs['date_utc'], format='ISO8601').dt.tz_convert('Europe/Moscow')
df_msgs['dt'] = df_msgs['dt'].dt.tz_localize(None)
df_msgs.drop(columns=['date_utc'], inplace=True)
df_msgs.set_index('dt', inplace=True)
# Фильтрация записей за период
mask = (df_msgs.index > '2022-08-01 00:00:00') & \
       (df_msgs.index < '2026-08-01 00:00:00')
df_msgs = df_msgs[mask]
df_msgs.info()
df_msgs.head()

<class 'pandas.DataFrame'>
DatetimeIndex: 171855 entries, 2026-07-31 22:06:20 to 2022-08-01 07:19:42
Data columns (total 1 columns):
 #   Column   Non-Null Count   Dtype
---  ------   --------------   -----
 0   message  171855 non-null  str  
dtypes: str(1)
memory usage: 2.6 MB


,message
dt,
2026-07-31 22:06:20,"🇮🇷#иран #today #трамп \nТрамп заявил, что Уитк..."
2026-07-31 21:25:15,🚫✴️🇷🇺#майнинг #россия #крипто\nПравительство Р...
2026-07-31 21:22:45,"🇺🇸🇷🇺#санкции #россия \nЗаконопроект об ""адских..."
2026-07-31 21:04:08,❗️🇯🇵 #fx #япония #интервенции\nСША и Япония пр...
2026-07-31 20:58:52,"⚠️🇷🇺#акции #россия \nПо состоянию на 21 июля, ..."


Датасет сообщений новостного канала содержит 171855 сообщений в поле message за период с 07:19 1 августа 2022 года по 22:06 31 июля 2026 года. Сообщения представлены в текстовом виде и не могут быть переданы для обучения в текущем виде, требуется проведение их обработки и кодирования.

## Предварительная подготовка данных

### Сообщения новостного канала

Сообщения новостного канала требуют обработки и выделения отдельных признаков.  
Часть сообщений содержат хештеги, выделим их в отдельный признак tags, текст сообщения без тегов поместим в признак text.

In [12]:
reg = r'#\w+'
df_msgs['tags'] = df_msgs['message'].str.findall(reg)
df_msgs['text'] = df_msgs['message'].str.replace(r'#\w+', '', regex=True)
df_msgs.head()

,message,tags,text
dt,,,
2026-07-31 22:06:20,"🇮🇷#иран #today #трамп \nТрамп заявил, что Уитк...","[#иран, #today, #трамп]","🇮🇷 \nТрамп заявил, что Уиткофф и Кушнер не п..."
2026-07-31 21:25:15,🚫✴️🇷🇺#майнинг #россия #крипто\nПравительство Р...,"[#майнинг, #россия, #крипто]",🚫✴️🇷🇺 \nПравительство РФ ввело запрет на майн...
2026-07-31 21:22:45,"🇺🇸🇷🇺#санкции #россия \nЗаконопроект об ""адских...","[#санкции, #россия]","🇺🇸🇷🇺 \nЗаконопроект об ""адских санкциях"" прот..."
2026-07-31 21:04:08,❗️🇯🇵 #fx #япония #интервенции\nСША и Япония пр...,"[#fx, #япония, #интервенции]",❗️🇯🇵 \nСША и Япония проводят интервенции в J...
2026-07-31 20:58:52,"⚠️🇷🇺#акции #россия \nПо состоянию на 21 июля, ...","[#акции, #россия, #ROSN, #SBER, #MTSS, #NMTP, ...","⚠️🇷🇺 \nПо состоянию на 21 июля, JPMorgan ликв..."


In [13]:
df_msgs['count_tags'] = df_msgs['tags'].apply(lambda x: len(x))

# Построение распределения количества тегов
plot_numerical_feature(df_msgs, 'count_tags', 'Количество хештегов')

Статистические данные признака:


count    171855.00
mean          2.69
std           1.51
min           0.00
25%           2.00
50%           3.00
75%           3.00
max          67.00
Name: count_tags, dtype: float64

Количество тегов в сообщениях канала распределено от 0 до 67, при этом, исходя из графика, сообщения, в которых количество хештегов более 4, очистим как выбросы.
Также, исходя из наблюдений за публикациями канала, нетегированные сообщения не несут в себе существенной информации, они в основном рекламные или освещают незначительные события, соответственно также подлежат удалению.

In [14]:
mask = (df_msgs['count_tags'] > 0) & (df_msgs['count_tags'] <= 4)
df_msgs = df_msgs[mask]
len(df_msgs)

157235

После очистки осталось 157235 сообщений.  
Проверим наличие сообщений, состоящих только из тегов, это могут быть сообщения, в которых информация приведена в прикрепленных мультимедиа (не обрабатываемая по условиям задачи).

In [17]:
df_msgs['text'] = df_msgs['text'].str.replace(r'[^a-zA-Z0-9а-яА-ЯёЁ]', '', regex=True)
len(df_msgs[df_msgs['text'] == ''])

3752

Таких сообщений 3752, очистим и от них, также удалим временный столбец text.

In [18]:
mask = df_msgs['text'] != ''
df_msgs = df_msgs[mask]
df_msgs.drop(columns=['text'], inplace=True)

Также в канале присутствуют "напоминающие" сообщения, включающие символы "❗️ВПЕРЕДИ", они не несут в себе информации, влияющей на цену актива, поэтому подлежат удалению.

In [19]:
mask = df_msgs['message'].str.contains('❗️ВПЕРЕДИ', na=False)
index_del = df_msgs[mask].index
df_msgs.drop(index=index_del, inplace=True)
df_msgs.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 153452 entries, 2026-07-31 22:06:20 to 2022-08-01 07:33:34
Data columns (total 3 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   message     153452 non-null  str   
 1   tags        153452 non-null  object
 2   count_tags  153452 non-null  int64 
dtypes: int64(1), object(1), str(1)
memory usage: 4.7+ MB


Таким образом после предварительной очистки осталось 153452 сообщений.  

Произведем оценку частоты использования хештегов.

In [20]:
all_tags = df_msgs['tags'].explode().unique().tolist()
len(all_tags)

4763

Всего в датасете встречается 4763 различных тега. Часть из них потенциально могут быть выделены в качестве отдельных признаков, остальные могут быть использованы как признак "в совокупности".

In [21]:
# Формирование статистики использования хештегов
dict_tags = {}
for tag in all_tags:
    dict_tags[tag] = df_msgs['tags'].apply(lambda x: tag in x).sum()
# Сортировка значений
dict_tags = dict(sorted(dict_tags.items(), key=lambda item: item[1], reverse=True))

In [22]:
# Построение графика
count_bar = 80                                                                      
# Инициализация полотна
fig_col = go.Figure(data=go.Bar(x=list(dict_tags.keys())[:count_bar],
                                y=list(dict_tags.values())[:count_bar]))
# Размер полотна
fig_col.update_layout(autosize = True)
# Заголовок диаграммы
fig_col.update_layout(title=f"Статистика использования хештегов",
                        title_x=0.5, # Расположение посередине
                        xaxis_title='Хештег', # Подпись Х
                        yaxis_title='Количество') # Подпись Y
# fig_col.show('png')
fig_col.show()

Исходя из частоты использования хештегов и отмечаемых особенностей финансового актива (золота), в качестве отдельных признаков выделяем:
- геополитика;
- отчетность;
- прогноз;
- экономика;
- дкп;
- инфляция;
- отчетности;
- золото.

К ним же добавим теги наиболее популярных стран/регионов:
- россия;
- сша;
- китай;
- европа;
- украина;
- иран;
- британия;
- япония;
- индия;
- германия.

Списки тегов оставляем в качестве отдельного признака. Так как при сравнении списков на равенство важен порядок элементов, то после выделения тегов сообщения в списки будет проводиться их сортировка.

In [23]:
col_tags = ['#геополитика', '#отчетность', '#прогноз', '#экономика',
            '#дкп', '#инфляция', '#отчетности', '#золото',
            '#россия', '#сша', '#китай', '#европа', '#украина', 
            '#иран', '#британия', '#япония', '#индия', '#германия']
# Формирование признаков наличия тега из списка в сообщении
for ct in col_tags:
    df_msgs[ct] = df_msgs['tags'].apply(lambda x: ct in x)
# Сортировка списков тегов
df_msgs['tags'] = df_msgs['tags'].apply(lambda x: sorted(x))

In [24]:
df_msgs['tags_comb'] = df_msgs['tags'].apply(lambda x: ''.join(x))
df_msgs['tags_comb'].nunique()

32875

In [25]:
# Формирование статистики использования сочетаний хештегов
dict_tags_comb = {}
for tag in df_msgs['tags_comb'].unique():
    dict_tags_comb[tag] = df_msgs['tags_comb'].apply(lambda x: tag in x).sum()
# Сортировка значений
dict_tags_comb = dict(sorted(dict_tags_comb.items(), key=lambda item: item[1], reverse=True))

In [26]:
# Построение графика
count_bar = 100                                                                      
# Инициализация полотна
fig_col = go.Figure(data=go.Bar(x=list(dict_tags_comb.keys())[:count_bar],
                                y=list(dict_tags_comb.values())[:count_bar]))
# Размер полотна
fig_col.update_layout(autosize = True)
# Заголовок диаграммы
fig_col.update_layout(title=f"Статистика использования сочетаний хештегов",
                        title_x=0.5, # Расположение посередине
                        xaxis_title='Сочетание хештегов', # Подпись Х
                        yaxis_title='Количество') # Подпись Y
# fig_col.show('png')
fig_col.show()

В датасете присутствует 32875 сочетаний хештегов. В качестве признаков оставим те сочетания, которые встречаются в 95% сообщений, остальные пометим как "другие", в эту же категорию будут попадать сочетания хештегов новых сообщений, в режиме работы на реальных данных. Данный признак планируется кодировать посредством Binary Encoding, поэтому даже при максимальном количестве сочетаний добавится всего 16 признаков, что приемлемо в условиях задачи.

In [27]:
def get_top_cat(freq_dict, share=0.99):
    # Сортирровка по убыванию частоты
    sorted_items = sorted(freq_dict.items(), key=lambda x: x[1], reverse=True)
    
    total = sum(freq_dict.values())
    cumulative = 0
    result_tags = []
    
    for tag, count in sorted_items:
        cumulative += count
        result_tags.append(tag)
        if cumulative / total >= share:
            break
    
    return result_tags

In [ ]:
# Получение списка сочетаний тегов, обеспечивающего покрытие 95% сообщений
top_tags_comb = get_top_cat(dict_tags_comb, 0.95)
len(top_tags_comb)

8663

В 95% сообщений встречается 8663 сочетания хештегов + 1 категория для остальных, что при Binary Encoding создаст дополнительно 14 признаков.

В качестве признака текст сообщения в чистом виде не может быть использован, для получения тональности сообщения использование стандартных библиотек не рекомендуется, так как в финансовых текстах возможны ошибки, потому что происходит оценка слов изолированно, без учета контекста. Из существующих анализаторов финансовых текстов наиболее приемлемый вариант использовать модель FinBERT библиотеки Hugging Face Transformers ввиду ее доступности и относительно малых вычислительных требований.

In [30]:
# Получение тональности сообщений
# Настройка устройства
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Используемое устройство: {device}")
if device == "cuda":
    print(f"Видеокарта: {torch.cuda.get_device_name(0)}")
    print(f"CUDA версия (runtime): {torch.version.cuda}")
else:
    print("!!!GPU не найден. Работа будет идти на CPU (медленнее).")
# Загрузка модели и токенизатора
model_name = "ProsusAI/finbert"
print("Загрузка модели FinBERT...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name,
                                                           use_safetensors=True)
# Перенс модели на устройство
model.to(device)
model.eval()  # Режим оценки (отключает dropout и т.д.)
# Категории тональности сообщений
labels = ["negative", "neutral", "positive"]

# Подготовка данных
texts = df_msgs['message'].tolist()
# Сохранение индексов
original_indices = df_msgs['message'].index.tolist()

print(f"Сообщений для анализа: {len(texts)}")
# Пакетная обработка (BATCHING) с GPU
batch_size = 128  # Варианты для GPU: 16, 32, 64, 128. Если Out Of Memory - уменьшать значение.
n_batches = math.ceil(len(df_msgs) / batch_size)

all_probs = []

print("Запуск инференса на GPU...")

for i in tqdm(range(n_batches), desc="FinBERT Inference"):
    batch_texts = texts[i*batch_size:(i+1)*batch_size]
    # Токенизация
    inputs = tokenizer(batch_texts,
                       return_tensors="pt",
                       truncation=True,
                       padding=True,
                       max_length=512)
    # Перенос тензоров на GPU
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
    # Получение вероятности
    probs = F.softmax(logits, dim=-1)
    all_probs.append(probs)
    
# Объединение батчей в один тензор и перенос на CPU для работы с pandas
all_probs_tensor = torch.cat(all_probs, dim=0)
all_probs_np = all_probs_tensor.cpu().numpy()

# Формирование результатов
pred_ids = np.argmax(all_probs_np, axis=1)
pred_labels = [labels[i] for i in pred_ids]
pred_conf = all_probs_np[np.arange(len(pred_ids)), pred_ids]

# Создание DataFrame с результатами
df_sent = pd.DataFrame({"prob_neg": all_probs_np[:, 0],
                        "prob_neu": all_probs_np[:, 1],
                        "prob_pos": all_probs_np[:, 2],
                        "sentiment": pred_labels,
                        "confidence": pred_conf})
# Восстановление исходных индексов
df_sent.index = original_indices
# Слияние с исходным датасетом
df_msgs_full = pd.concat([df_msgs, df_sent], axis=1)

Используемое устройство: cuda
Видеокарта: NVIDIA GeForce RTX 4070 Laptop GPU
CUDA версия (runtime): 12.1
Загрузка модели FinBERT...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 5977.38it/s]


Сообщений для анализа: 153452
Запуск инференса на GPU...


FinBERT Inference: 100%|██████████| 1199/1199 [29:19<00:00,  1.47s/it]


In [31]:
df_msgs_full.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 153452 entries, 2026-07-31 22:06:20 to 2022-08-01 07:33:34
Data columns (total 27 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   message       153452 non-null  str    
 1   tags          153452 non-null  object 
 2   count_tags    153452 non-null  int64  
 3   #геополитика  153452 non-null  bool   
 4   #отчетность   153452 non-null  bool   
 5   #прогноз      153452 non-null  bool   
 6   #экономика    153452 non-null  bool   
 7   #дкп          153452 non-null  bool   
 8   #инфляция     153452 non-null  bool   
 9   #отчетности   153452 non-null  bool   
 10  #золото       153452 non-null  bool   
 11  #россия       153452 non-null  bool   
 12  #сша          153452 non-null  bool   
 13  #китай        153452 non-null  bool   
 14  #европа       153452 non-null  bool   
 15  #украина      153452 non-null  bool   
 16  #иран         153452 non-null  bool   
 17  #британия     153452 non-

In [32]:
# Краткая статистика
print("\nСтатистика тональности:")
print(df_msgs_full["sentiment"].value_counts())
print(f"Средняя уверенность модели: {df_msgs_full['confidence'].mean():.3f}")


Статистика тональности:
sentiment
positive    150975
negative      1603
neutral        874
Name: count, dtype: int64
Средняя уверенность модели: 0.835


В результате обработки текста сообщений моделью FinBERT получены следующие признаки:
- prob_neg, prob_neu, prob_pos - вероятности текста быть финансово негативной, нейтральной или позитивной новостью;
- sentiment - класс новости (с наибольшей вероятностью);
- confidence - вероятность класса новости.

Признаки sentiment и confidence для модели избыточны и фактически вытекают из prob_neg, prob_neu и prob_pos, а, соответственно, подлежат удалению.  
Из признаков prob_neg, prob_neu и prob_pos можно получить сверточный признак sent_score, равный разнице вероятностей позитивной и негативной тональности, таким образом мы получим тональность одним числом от -1 до 1, где -1 это абсолютно негативный тон, 0 - нейтральный, а 1 - абсолютно позитивный.  
Для выбранных алгоритмов машинного обучения оставим и три вероятности, и sent_score, данная избыточность невелика, но может дать лучший вариант при обучении.

Также сразу удалим признаки:
- message - обработан, получены новые признаки;
- tags - обработан, получены новые признаки;
- count_tags - не имеет информационного значения для целевых признаков.

In [ ]:
df_msgs_full.drop(columns=['sentiment','confidence', 'message', 'tags', 'count_tags'],
                  inplace=True)
df_msgs_full['sent_score'] = df_msgs_full['prob_pos'] - df_msgs_full['prob_neg']

Проведем кодирование признака tags_comb с использованием полученного списка сочетаний тегов посредством бинарного кодирования.

In [ ]:
# Изменение значения сочетаний тегов на "other" для невошедших в список
df_msgs_full['tags_comb'] = df_msgs_full['tags_comb'].apply(lambda x: 'other' if x not in top_tags_comb else x)
# Бинарное кодирование признака
encoder_tags_comb = ce.BinaryEncoder(cols=['tags_comb'])
encoded_tc = encoder_tags_comb.fit_transform(df_msgs_full[['tags_comb']])
print(f'Закодировано в {encoded_tc.shape[1]} признаках')
df_msgs_full = pd.concat([df_msgs_full, encoded_tc], axis=1)
# Удаление признака tags_comb
df_msgs_full.drop(columns=['tags_comb'], inplace=True)

In [41]:
df_msgs_full.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 153452 entries, 2026-07-31 22:06:20 to 2022-08-01 07:33:34
Data columns (total 36 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   #геополитика  153452 non-null  bool   
 1   #отчетность   153452 non-null  bool   
 2   #прогноз      153452 non-null  bool   
 3   #экономика    153452 non-null  bool   
 4   #дкп          153452 non-null  bool   
 5   #инфляция     153452 non-null  bool   
 6   #отчетности   153452 non-null  bool   
 7   #золото       153452 non-null  bool   
 8   #россия       153452 non-null  bool   
 9   #сша          153452 non-null  bool   
 10  #китай        153452 non-null  bool   
 11  #европа       153452 non-null  bool   
 12  #украина      153452 non-null  bool   
 13  #иран         153452 non-null  bool   
 14  #британия     153452 non-null  bool   
 15  #япония       153452 non-null  bool   
 16  #индия        153452 non-null  bool   
 17  #германия     153452 non-

Таким образом в ходе обработки сообщений новостного канала получен датасет, включающий в себя 153452 записи по 36 признакам. Данные обработаны и закодированы и могут быть использованы для обучения ML-моделей без дополнительной существенной обработки.

---

### Вычисление факторов на базе котировок

Вычисление значений индикаторов на 15-минутном графике.

In [ ]:
# EMA, период 9
df_main_period['EMA_fast'] = ta.ema(df_main_period['close'], length=period_EMA_fast)
# EMA, период 21
df_main_period['EMA_slow'] = ta.ema(df_main_period['close'], length=period_EMA_slow)
# VWAP, с начала торговой сессии
df_main_period['VWAP'] = ta.vwap(df_main_period['high'], df_main_period['low'], df_main_period['close'], df_main_period['volume'])
# RSI, период 7
df_main_period['RSI'] = ta.rsi(df_main_period['close'], length=period_RSI)
# MACD, параметры (12, 26, 9)
MACD = ta.macd(df_main_period['close'], fast=period_MACD_fast, slow=period_MACD_slow, signal=period_MACD_signal)
added_name = '_' + str(period_MACD_fast) + '_' + str(period_MACD_slow) + '_' + str(period_MACD_signal)
df_main_period['MACD'] = MACD['MACD' + added_name]
df_main_period['MACD_signal'] = MACD['MACDs' + added_name]
df_main_period['MACD_hist'] = MACD['MACDh' + added_name]

In [ ]:
# Отбор данных
show_df = df_main_period.tail(150)
# Построение графика японских свечей с индикаторами
ema = fplt.make_addplot(show_df[['EMA_fast', 'EMA_slow', 'VWAP']])
rsi = fplt.make_addplot(show_df["RSI"], color="grey", width=1.5, ylabel="RSI",
                        secondary_y=True, linestyle='dashdot')
fplt.plot(show_df,
          type='candle',
          addplot=[ema, rsi],
          volume=True,
          style='charles',
          title='GOLD: последние 150 свечей датасета',
          ylabel='Price ($)',
          figsize=(15, 6))
fplt.show()

Согласно книги Стива Ниссона "Японские свечи" каждая свеча описывается следующими параметрами:
- тело свечи;
- верхняя и нижняя тени;
- является ли свеча растущей.

Дальнейшее выделение патернов проводится на основе комбинаций свечей, определяемых данными признаками, что позволяет предположить их значимость при формировании предсказательных моделей.  
В большинстве случаев производится категориальное разделение на патерны, что подводит к решению о целесообразности применения в качестве базового алгоритма построения предсказательной модели дерева решений, случайного леса.

In [ ]:
# Добавление признаков описания свечей
# Тело свечи
df_main_period['real_body'] = np.abs(df_main_period['close'] - df_main_period['open'])
# Является растущей свечой
df_main_period['is_growing_candle'] = df_main_period['close'] > df_main_period['open']
# Верхняя и нижняя тени
df_main_period['upper_shadow'] = np.where(df_main_period['is_growing_candle'], 
                                  df_main_period['high'] - df_main_period['close'],
                                  df_main_period['high'] - df_main_period['open'])
df_main_period['lower_shadow'] = np.where(df_main_period['is_growing_candle'], 
                                  df_main_period['high'] - df_main_period['open'],
                                  df_main_period['high'] - df_main_period['close'])
df_main_period.tail()

### Добавление значимых уровней

Для дейтрейдинга на M15 с целевой продолжительностью сделки в 30-90 минут принимают во внимание следующие уровни:
- Поддержка и сопротивление — локальные и важные уровни.
- Психологические уровни — круглые числа.
- Фибоначчи — уровни отката и расширения.
- Скользящие средние (SMA/EMA) — динамические уровни.
- VWAP — уровень средней цены с учетом объема.
- Пробойные уровни — ключевые уровни поддержки/сопротивления.
- Технические фигуры — паттерны на графиках.

Уровни поддержки и сопротивления, а также пробойные уровни относительно субъективны, скользящие средние и VWAP вычисляются в качестве технических индикаторов и присутствуют в наборе.

В качестве признаков добавим
- ближайшие круглые числа (для золота на 15-минутных свечах - кратные 5.0);
- уровни расширения Фибоначчи для восходящего и нисходящего тренда

In [ ]:
def fibonacci_extension_levels(df: pd.DataFrame, period: int, long_levels: int = 7, short_levels: int = 7) -> pd.DataFrame:
    """
    Рассчитывает уровни Фибоначчи расширения для восходящего и нисходящего тренда.
    Оптимизировано для производительности с использованием numpy и pandas.

    :param df: pd.DataFrame с колонками 'high', 'low'
    :param period: Количество свечей для расчета
    :param long_levels: Количество ближайших уровней для восходящего тренда
    :param short_levels: Количество ближайших уровней для нисходящего тренда
    :return: pd.DataFrame с уровнями Фибоначчи для каждой свечи
    """
    
    # Стандартные коэффициенты уровней расширения Фибоначчи
    extension_ratios = [1.0, 1.272, 1.414, 1.618, 2.0, 2.618, 3.618]
    
    # Ограничиваем количество уровней для каждого тренда
    long_ratios = extension_ratios[:long_levels]
    short_ratios = extension_ratios[:short_levels]
    
    # Преобразуем high и low в numpy массивы для ускорения вычислений
    high_values = df['high'].values
    low_values = df['low'].values
    
    # Создаем массивы для хранения уровней расширения для всех свечей
    long_levels_matrix = np.full((len(df), len(long_ratios)), np.nan)
    short_levels_matrix = np.full((len(df), len(short_ratios)), np.nan)
    
    # Вычисляем максимумы и минимумы для всех окон с использованием rolling
    max_high = df['high'].rolling(window=period, min_periods=period).max().values
    min_low = df['low'].rolling(window=period, min_periods=period).min().values
    
    # Векторизация расчета уровней Фибоначчи для каждого окна
    for i in range(period, len(df)):
        high = max_high[i]
        low = min_low[i]
        diff = high - low
        
        if diff == 0:
            continue  # Избегаем деления на ноль
        
        # Восходящий тренд: расширение от low вверх
        long_levels_matrix[i, :] = low + (np.array(long_ratios) - 1) * diff
        
        # Нисходящий тренд: расширение от high вниз
        short_levels_matrix[i, :] = high - (np.array(short_ratios) - 1) * diff
    
    # Собираем результаты в DataFrame
    result = pd.DataFrame(index=df.index)
    
    # Добавляем уровни для восходящего тренда (long)
    for idx, ratio in enumerate(long_ratios):
        result[f"long_ext_{ratio:.3f}"] = long_levels_matrix[:, idx]
    
    # Добавляем уровни для нисходящего тренда (short)
    for idx, ratio in enumerate(short_ratios):
        result[f"short_ext_{ratio:.3f}"] = short_levels_matrix[:, idx]

    return result

In [ ]:
# Вычисление уровней Фибоначчи
df_main_period = df_main_period.join(fibonacci_extension_levels(df=df_main_period, period=count_FE_levels))
# Добавление ближайших психологических уровней (круглые числа)
df_main_period['round_price_up'] = df_main_period['close']//round_price * round_price + round_price
df_main_period['round_price_down'] = df_main_period['close']//round_price * round_price

Отдельно во многих изданиях (например, Джон Мерфи "Технический анализ финансовых рынков") подчеркивается важность ценовых разрывов (gap). Сформируем признак "is_gapped_up", если свеча имеет разрыв вверх относительно предыдущей свечи, и "is_gapped_down" в случае разрыва вниз.

In [ ]:
df_main_period['is_gapped_up'] = df_main_period['low'] > df_main_period['high'].shift(1)
df_main_period['is_gapped_down'] = df_main_period['high'] < df_main_period['low'].shift(1)

После вычислений всех индикаторов и признаков производим округление значений до шага инструмента, в нашем случае **0.1**, кроме значений технических индикаторов, вычисляемых в масштабе, отличном от цены актива.

In [ ]:
df_main_period['is_gapped_up'].dtype

In [ ]:
list_sign_ta = ['RSI', 'MACD', 'MACD_signal', 'MACD_hist']

for col in df_main_period.columns:
    if (pd.api.types.is_float_dtype(df_main_period[col])):
        if col in list_sign_ta:
            df_main_period[col] = df_main_period[col].round(round_ta)
        else:
            df_main_period[col] = (df_main_period[col] + 0.5 * step_price)//step_price * step_price
for col in df_M1.columns:
    if (pd.api.types.is_float_dtype(df_M1[col])):
        df_M1[col] = (df_M1[col] + 0.5 * step_price)//step_price * step_price
        
df_main_period.info()
df_main_period

### Ретроспективность объектов

Ретроспективность объектов обеспечим добавлением признаков трех предыдущих свечей с добавлением пометки "_prev_1", "_prev_2" и "_prev_3" соответственно. На данный момент каждая свеча описывается следующими признаками:
- open - цена открытия;
- high - максимальная цена;
- low - минимальная цена;
- close - цена закрытия;
- volume - объем торгов;
- EMA_fast - значение быстрой EMA;
- EMA_slow - значение медленной EMA;
- VWAP - значение VWAP;
- RSI - значение RSI;
- MACD, MACD_signal, MACD_hist - значения MACD;
- real_body - тело свечи;
- is_growing_candle - является ли свеча растущей;
- upper_shadow - верхняя тень;
- lower_shadow - нижняя тень;
- long_ext_1.000, long_ext_1.272, long_ext_1.414, long_ext_1.618, long_ext_2.000, long_ext_2.618, long_ext_3.618 - уровни расширения Фибоначчи для восходящего тренда;
- short_ext_1.000, short_ext_1.272, short_ext_1.414, short_ext_1.618, short_ext_2.000, short_ext_2.618, short_ext_3.618 - уровни расширения Фибоначчи для нисходящего тренда;
- round_price_up - ближайшее круглое число сверху;
- round_price_down - ближайшее круглое число сверху;
- is_gapped_up - свеча сформировалась с разрывом вверх;
- is_gapped_down - свеча сформировалась с разрывом вниз.

При обеспечении ретроспективности данных не имеет смысла копировать признаки, отражающие значимые уровни, за исключением EMA, VWAP, так как помимо интерпретации данных значений как уровней, имеют значение также их предыдущие значения (направление линий индикаторов). Таким образом копированию подлежат следующие признаки:
- open,
- high,
- low,
- close,
- volume,
- EMA_fast,
- EMA_slow,
- VWAP,
- RSI,
- MACD,
- MACD_signal,
- MACD_hist,
- real_body,
- is_growing_candle,
- upper_shadow,
- lower_shadow,
- is_gapped_up,
- is_gapped_down.

In [ ]:
list_previous_candle_sign = ['open', 'high', 'low', 'close', 'volume', 
                             'EMA_fast', 'EMA_slow', 'VWAP', 'RSI', 'MACD', 'MACD_signal', 'MACD_hist', 
                             'real_body', 'is_growing_candle', 'upper_shadow', 'lower_shadow', 
                             'is_gapped_up', 'is_gapped_down']
for i in range(1, count_previous_candle + 1):
    for sign in list_previous_candle_sign:
        # Pandas при использовании shift на Series типа bool возвращает object, 
        # добавляем дополнительное преобразование
        if pd.api.types.is_bool_dtype( df_main_period[sign]):
            df_main_period[sign + '_prev_' + str(i)] = df_main_period[sign].shift(i).convert_dtypes()
        else:
            df_main_period[sign + '_prev_' + str(i)] = df_main_period[sign].shift(i)
df_main_period['is_growing_candle'] = df_main_period['is_growing_candle'].convert_dtypes()
df_main_period['is_gapped_up'] = df_main_period['is_gapped_up'].convert_dtypes()
df_main_period['is_gapped_down'] = df_main_period['is_gapped_down'].convert_dtypes()
df_main_period.info()

In [ ]:
# Выборочный контроль
df_main_period[['close', 'close_prev_1', 'close_prev_2', 'close_prev_3',
        'VWAP', 'VWAP_prev_1', 'VWAP_prev_2', 'VWAP_prev_3',
        'RSI', 'RSI_prev_1', 'RSI_prev_2', 'RSI_prev_3']]

In [ ]:
# Выборочный контроль
df_main_period[['is_gapped_up', 'is_gapped_up_prev_1', 'is_gapped_up_prev_2', 'is_gapped_up_prev_3']]

Дополнительно введем признак min_to_next (Минут до формирования свечи для пересчета основных патернов).

In [ ]:
# Округление вверх до ближайших 15 минут
df_M1['min_to_next'] = df_M1.index.ceil(main_period) - df_M1.index
df_M1['min_to_next'] = df_M1['min_to_next'].dt.components['minutes']

In [ ]:
df_M1['min_to_next'].head(10)

В конце месяца крупные фирмы могут фиксировать прибыль, выделим два признака - месяц и число месяца.

In [ ]:
# выделение числа и месяца
df_M1['month_day'] = df_M1.index.day
df_M1['month'] = df_M1.index.month
df_M1['hour'] = df_M1.index.hour

### Формирование целевых признаков

В качестве целевых признаков выбраны:
- цена закрытия в конце периода;
- максимальная цена за последующий период;
- минимальная цена за последующий период.

Выбранный период соответствует сроку удержания позиции - 60 минут. Поступление данных может быть неравномерным из-за разных источноков данных, например, несмотря на то, что основные финансовые отчеты публикуются обычно в строго определенное время, кратное 15 минутам (16:45, 17:30 и т.п.), появление их на канале может быть с задержкой. Поэтому рассчет целевых признаков будем производить на базе минутных свечей. Также цена потенциального исполнения рыночной заявки будет ближе к цене закрытия минутной свечи, чем к цене закрытия предыдущей 15-минутной. Соответственно итоговое сведение показателей технических индикаторов будет сводиться к цене закрытия минутных свечей.

In [ ]:
# Вычисляем максимумы и минимумы для всех окон с использованием rolling
df_M1['predict_close'] = df_M1['close'].shift(- period_holding_position)
df_M1['predict_high'] = df_M1['high'].iloc[::-1].rolling(window=period_holding_position, min_periods=period_holding_position).max().values[::-1]
df_M1['predict_low'] = df_M1['low'].iloc[::-1].rolling(window=period_holding_position, min_periods=period_holding_position).min().values[::-1]
df_M1['predict_to_up'] = ((df_M1['predict_close'] - df_M1['close'])/df_M1['close'] * 100) >= level_price_movement_direction
df_M1['predict_to_down'] = ((df_M1['predict_close'] - df_M1['close'])/df_M1['close'] * 100) <= -level_price_movement_direction
df_M1.drop(['predict_close'], axis=1, inplace=True)
df_M1[['predict_high', 'predict_low', 'predict_to_up', 'predict_to_down']]

In [ ]:
# Сокращение используемой памяти
df_M1 = df_M1.astype({col: 'float32' for col in df_M1.select_dtypes(include='float64').columns})
df_main_period = df_main_period.astype({col: 'float32' for col in df_main_period.select_dtypes(include='float64').columns})
df_M1.info()
df_main_period.info()

### Объединение датасетов df_M1 и df_main_period

In [ ]:
# Приведение индексов к реальному времени формирования данных
df_M1_real_time = df_M1.copy()
df_main_period_real_time = df_main_period.copy()

df_M1_real_time.index = df_M1_real_time.index + pd.DateOffset(minutes=1)
df_main_period_real_time.index = df_main_period_real_time.index + pd.DateOffset(minutes=main_period_int)

# Сортировка по индексам
df_M1_real_time = df_M1_real_time.sort_index()
df_main_period_real_time = df_main_period_real_time.sort_index()

# Объединение
df_result = pd.merge_asof(df_M1_real_time, 
                          df_main_period_real_time,
                          left_index=True,
                          right_index=True,
                          direction='backward')

df_result.info()

После объединения датасетов требуется приведение типов отдельных признаков для оптимизации использования памяти.

In [ ]:
for col in df_result.columns:
    if col.find('volume') >= 0:
        df_result[col] = df_result[col].fillna(0)
        df_result[col] = df_result[col].astype('int32')
    if col == 'min_to_next':
        df_result[col] = df_result[col].astype('int32')
df_result.info()

Кодирование дней недели проведем методом однократного кодирования.

In [ ]:
# Кодирование категориальных признаков не требуется при использовании CatBoost
# encoder_day_of_week = ce.OneHotEncoder(cols=['day_of_week'],
#                                        use_cat_names=True)
# day_of_week_bin = encoder_day_of_week.fit_transform(df_result['day_of_week'])

# df_result = pd.concat([df_result, day_of_week_bin], axis=1)
# df_result.drop(columns=['day_of_week'],
#                axis=1,
#                inplace=True)
# df_result.info()

После однократного кодирования требуется приведение типов полученных признаков к bool для оптимизации использования памяти.

In [ ]:
# for col in df_result.columns:
#     if col.find('day_of_week_') >= 0:
#         df_result[col] = df_result[col].astype('boolean')
# df_result.info()

В качестве целевого признака модели для определения значения TakeProfit выбираем уровень, максимально удаленный от цены закрытия, но который пересекает цена в течении периода сделки (меньше максимальной или больше минимальной).

In [ ]:
# dict_levels = {'EMA_fast': 0, 'EMA_slow': 1, 'VWAP': 2, 
#                'long_ext_1.000': 3, 'long_ext_1.272': 4, 'long_ext_1.414': 5, 'long_ext_1.618': 6, 
#                'long_ext_2.000': 7, 'long_ext_2.618': 8, 'long_ext_3.618': 9,
#                'short_ext_1.000': 10, 'short_ext_1.272': 11, 'short_ext_1.414': 12, 'short_ext_1.618': 13, 
#                'short_ext_2.000': 14, 'short_ext_2.618': 15, 'short_ext_3.618': 16}

In [ ]:
dict_levels = {'EMA_fast': 0, 'EMA_slow': 1, 'VWAP': 2, 
               'long_ext_1.000': 3, 'long_ext_1.272': 4, 'long_ext_1.414': 5, 'long_ext_1.618': 6, 
               'long_ext_2.000': 7, 'long_ext_2.618': 8, 'long_ext_3.618': 9,
               'short_ext_1.272': 10, 'short_ext_1.414': 11, 'short_ext_1.618': 12, 
               'short_ext_2.618': 13, 'short_ext_3.618': 14}

In [ ]:
def add_predict_level(df: pd.DataFrame,
                      price_col: str = "close_x",
                      high_col: str = "predict_high",
                      low_col: str = "predict_low",
                      level_columns: list | None = None,
                      inplace: bool = False) -> pd.DataFrame:
    """
    Добавляет столбец `predict_level` – название уровня цены,
    который максимально отклонён от цены закрытия, но находится
    внутри предсказанного диапазона [predict_low, predict_high].

    Parameters
    ----------
    df : pd.DataFrame
        Исходный дата‑фрейм, содержащий как минимум столбцы:
        `price_col`, `high_col`, `low_col` и все `level_columns`.
    price_col : str, default "close_x"
        Столбец с ценой закрытия.
    high_col : str, default "predict_high"
        Верхняя граница предсказанного диапазона.
    low_col : str, default "predict_low"
        Нижняя граница предсказанного диапазона.
    level_columns : list | None
        Список названий столбцов‑уровней. Если ``None`` – берётся
        предопределённый набор из условия задачи.
    inplace : bool, default False
        Если ``True`` – модифицирует `df` на месте и возвращает его.
        Иначе создаётся копия.

    Returns
    -------
    pd.DataFrame
        Дата‑фрейм с новым столбцом `predict_level`.
    """
    # ------------------------------------------------------------------
    # 0. Подготовка списка уровней (если пользователь не передал)
    # ------------------------------------------------------------------
    if level_columns is None:
        level_columns = [
            "EMA_fast", "EMA_slow", "VWAP",
            "long_ext_1.000", "long_ext_1.272", "long_ext_1.414",
            "long_ext_1.618", "long_ext_2.000", "long_ext_2.618",
            "long_ext_3.618",
            "short_ext_1.000", "short_ext_1.272", "short_ext_1.414",
            "short_ext_1.618", "short_ext_2.000", "short_ext_2.618",
            "short_ext_3.618",
        ]

    # ------------------------------------------------------------------
    # 1. Делать всё в копии или на месте?
    # ------------------------------------------------------------------
    if not inplace:
        df = df.copy()

    # ------------------------------------------------------------------
    # 2. Выделяем массивы (numpy) – это гораздо быстрее, чем apply
    # ------------------------------------------------------------------
    # (n_rows, n_levels) – значения всех уровней
    level_vals = df[level_columns].values.astype(float)

    # (n_rows, 1) – границы диапазона
    low_vals  = df[low_col].values[:, None].astype(float)
    high_vals = df[high_col].values[:, None].astype(float)

    # ------------------------------------------------------------------
    # 3. Маска: уровень попадает в диапазон [low, high] ?
    # ------------------------------------------------------------------
    inside_mask = (level_vals >= low_vals) & (level_vals <= high_vals)

    # ------------------------------------------------------------------
    # 4. Абсолютные отклонения от цены закрытия
    # ------------------------------------------------------------------
    close_vals = df[price_col].values[:, None].astype(float)
    diff = np.abs(level_vals - close_vals)

    # Для уровней, которые **не** попадают в диапазон, ставим -inf,
    # чтобы они никогда не выиграли в argmax.
    diff[~inside_mask] = -np.inf

    # ------------------------------------------------------------------
    # 5. Находим индекс уровня с максимальной разницей
    # ------------------------------------------------------------------
    best_idx = diff.argmax(axis=1)               # (n_rows,)

    # Проверяем, есть ли хотя бы один допустимый уровень в строке
    any_inside = inside_mask.any(axis=1)          # (n_rows,)

    # Преобразуем индексы в имена колонок
    best_names = np.array(level_columns)[best_idx]

    # Если в строке нет ни одного уровня внутри диапазона → NaN
    # result = np.where(any_inside, best_names, np.nan)
    result = np.where(any_inside, best_names, '')

    # ------------------------------------------------------------------
    # 6. Записываем результат в дата‑фрейм
    # ------------------------------------------------------------------
    df["predict_level"] = result

    return df

In [ ]:
df_result = add_predict_level(df_result, level_columns=list(dict_levels.keys()), inplace=True)

In [ ]:
# Кодирование классов уровней
df_result['predict_level'] = df_result['predict_level'].apply(lambda x: dict_levels.get(x))
df_result['predict_level'] =  df_result['predict_level'].astype('Int64')

In [ ]:
# df_result.to_csv('./data/df_result.csv')

> Таким образом на данном этапе датафрейм df_result содержит на минутном интервале:
> - текущее значения открытия, максимума, минимума, закрытия и объема минутной свечи (open_x, high_x, low_x, close_x, volume_x);
> - прогнозируемые значения (predict_close, predict_high, predict_low);
> - текущее значения открытия, максимума, минимума, закрытия и объема последней 15-минутной свечи (open_y, high_y, low_y, close_y, volume_y);
> - текущие значения выбранных технических индикаторов (15-минутное основание);
> - значимые уровни цен;
> - ретроспективные значения 15-минутных свечей и технических индикаторов;
> - закодированный день недели.
> 
> Датасет требует очистки.

## Очистка итогового датасета

Датасет df_result требует очистки от пустых значений и от недостоверных данных. Очистку производить путем удаления, пустые значения образованы из-за применения оконных функций и вычисления целевых признаков. Недостоверные данные образованы на склейке фьючерсных контрактов.

### Очистка пропущенных значений

In [ ]:
df_result.dropna(axis=0, how='any', inplace=True)
df_result.info()

# Проверка на пропущенные значения
print(f'Количество пропущенных значений: {df_result.isnull().sum().sum()}')

### Получение дат экспирации фьючерсных контрактов

Информацию о датах экспирации фьчерсных контрактов на золото гможно получить на сайте [Московской биржи](https://www.moex.com/ru/derivatives/contracts.aspx). Параметры поиска:
- Строка поиска: GOLD
- Искать контракты: Все
- Тип контракта: Фьючерсы
- Сортировать по: Дате исполнения

Анализ дат для удаления произведен вручную, так как для проверки необходим просмотр всего порядка 75 дат, следующих за датой окончания обращения фьючерса, а разработка алгоритма определения их затруднена ввиду проведения торговли только в рабочие дни и требует учета не только выходных, но и праздничных дней.

В качестве инструмента для анализа данных выбран TSLab.

Удалению подлежат данные за два дня до последнего дня обращения (из-за перехода трейдеров в следующий фьючерс или закрытие позиций по текущему) и следующий торговый день после, так как склейка фьючерсов некорректно отражается на значениях технических индикаторов при гэпе цен, образованных в результате склеек фьючерсов, максимальный текущий период равен 26, что на 15-минутных свечах составляет шесть с половиной часов.

In [ ]:
dates_for_del = ['12.03.2009', '13.03.2009', '16.03.2009', '10.06.2009', '11.06.2009', '15.06.2009',
                 '11.09.2009', '14.09.2009', '15.09.2009', '11.12.2009', '14.12.2009', '15.12.2009', 
                 '11.03.2010', '12.03.2010', '15.03.2010', '10.06.2010', '11.06.2010', '15.06.2010', 
                 '13.09.2010', '14.09.2010', '15.09.2010', '13.12.2010', '14.12.2010', '15.12.2010', 
                 '11.03.2011', '14.03.2011', '15.03.2011', '14.06.2011', '15.06.2011', '16.06.2011', 
                 '14.09.2011', '15.09.2011', '16.09.2011', '14.12.2011', '15.12.2011', '16.12.2011', 
                 '14.03.2012', '15.03.2012', '16.03.2012', '14.06.2012', '15.06.2012', '18.06.2012', 
                 '14.09.2012', '17.09.2012', '18.09.2012', '14.12.2012', '17.12.2012', '18.12.2012', 
                 '14.03.2013', '15.03.2013', '18.03.2013', '14.06.2013', '17.06.2013', '18.06.2013', 
                 '13.09.2013', '16.09.2013', '17.09.2013', '13.12.2013', '16.12.2013', '17.12.2013', 
                 '14.03.2014', '17.03.2014', '18.03.2014', '13.06.2014', '16.06.2014', '17.06.2014', 
                 '12.09.2014', '15.09.2014', '16.09.2014', '12.12.2014', '15.12.2014', '16.12.2014', 
                 '13.03.2015', '16.03.2015', '17.03.2015', '11.06.2015', '15.06.2015', '16.06.2015', 
                 '14.09.2015', '15.09.2015', '16.09.2015', '14.12.2015', '15.12.2015', '16.12.2015', 
                 '16.03.2016', '17.03.2016', '18.03.2016', '15.06.2016', '16.06.2016', '17.06.2016', 
                 '13.09.2016', '15.09.2016', '16.09.2016', '14.12.2016', '15.12.2016', '16.12.2016', 
                 '15.03.2017', '16.03.2017', '17.03.2017', '14.06.2017', '15.06.2017', '16.06.2017', 
                 '20.09.2017', '21.09.2017', '22.09.2017', '20.12.2017', '21.12.2017', '22.12.2017', 
                 '14.03.2018', '15.03.2018', '16.03.2018', '20.06.2018', '21.06.2018', '22.06.2018', 
                 '19.09.2018', '20.09.2018', '21.09.2018', '19.12.2018', '20.12.2018', '21.12.2018', 
                 '20.03.2019', '21.03.2019', '22.03.2019', '19.06.2019', '20.06.2019', '21.06.2019', 
                 '18.09.2019', '19.09.2019', '20.09.2019', '18.12.2019', '19.12.2019', '20.12.2019', 
                 '18.03.2020', '19.03.2020', '20.03.2020', '17.06.2020', '18.06.2020', '19.06.2020', 
                 '16.09.2020', '17.09.2020', '18.09.2020', '16.12.2020', '17.12.2020', '18.12.2020', 
                 '17.03.2021', '18.03.2021', '19.03.2021', '16.06.2021', '17.06.2021', '18.06.2021', 
                 '15.09.2021', '16.09.2021', '17.09.2021', '15.12.2021', '16.12.2021', '17.12.2021', 
                 '16.03.2022', '17.03.2022', '18.03.2022', '15.06.2022', '16.06.2022', '17.06.2022', 
                 '15.09.2022', '16.09.2022', '19.09.2022', '15.12.2022', '16.12.2022', '19.12.2022', 
                 '16.03.2023', '17.03.2023', '20.03.2023', '15.06.2023', '16.06.2023', '19.06.2023', 
                 '21.09.2023', '22.09.2023', '25.09.2023', '21.12.2023', '22.12.2023', '25.12.2023', 
                 '21.03.2024', '22.03.2024', '25.03.2024', '20.06.2024', '21.06.2024', '24.06.2024', 
                 '19.09.2024', '20.09.2024', '23.09.2024', '19.12.2024', '20.12.2024', '23.12.2024',
                 '20.03.2025', '21.03.2025', '24.03.2025', '19.06.2025', '20.06.2025', '23.06.2025', 
                 '18.09.2025', '19.09.2025', '22.09.2025', '18.12.2025', '19.12.2025', '22.12.2025', 
                 '19.03.2026', '20.03.2026', '23.03.2026', '18.06.2026', '19.06.2026', '22.06.2026',
                 '17.09.2026', '18.09.2026', '21.09.2026', '17.12.2026', '18.12.2026', '21.12.2026', 
                 '18.03.2027', '19.03.2027', '22.03.2027', '17.06.2027', '18.06.2027', '21.06.2027']

In [ ]:
# Удаление строк, попадающие в даты
df_result = df_result[~df_result.index.normalize().isin(pd.to_datetime(dates_for_del, format='%d.%m.%Y'))]
df_result.info()

Так как выбранное время торговли ограничено 10:30-17:30 с перерывом на клиринг 14:00-14:05, соответственно из анализа исключаются записи, выходящие за данный диапзон.

In [ ]:
# Выделение времени из индекса
time_index = df_result.index.time
# Маска по времени
mask = (((time_index >= pd.to_datetime('10:30').time()) & (time_index <= pd.to_datetime('13:59').time())) |
        ((time_index >= pd.to_datetime('14:06').time()) & (time_index <= pd.to_datetime('17:30').time())))
# Фильтрация по маске
df_result = df_result[mask]
df_result.info()

Из всех сформированных признаков в масштабе цены имеют значения все, кроме:
- иникатора RSI;
- объемов продаж;
- признаков растущей цены;
- признаков гэпа;
- дня недели проведения торгов.

По всем остальным неоходимо проведение масштабирования относительно текущей цены закрытия, при этом для повышения точности перевоим в формат процентов с округлением до трех знаков после разделителя.

In [ ]:
# Список признаков, исключаемых из масштабирования
exclude_futures = ['volume_x', 'volume_y', 'RSI', 'is_growing_candle', 'is_gapped_up', 'is_gapped_down', 
                   'volume_prev_1', 'RSI_prev_1', 'is_growing_candle_prev_1', 'is_gapped_up_prev_1', 'is_gapped_down_prev_1',
                   'volume_prev_2', 'RSI_prev_2', 'is_growing_candle_prev_2', 'is_gapped_up_prev_2', 'is_gapped_down_prev_2',
                   'volume_prev_3', 'RSI_prev_3', 'is_growing_candle_prev_3', 'is_gapped_up_prev_3', 'is_gapped_down_prev_3', 
                   'day_of_week', 'min_to_next', 'close_x',
                   'predict_to_up', 'predict_to_down', 'predict_level', 'month_day', 'month', 'hour']
without_div = ['MACD', 'MACD_signal', 'MACD_hist', 'real_body', 'upper_shadow', 'lower_shadow',
               'MACD_prev_1', 'MACD_signal_prev_1', 'MACD_hist_prev_1', 'real_body_prev_1', 
               'is_growing_candle_prev_1', 'upper_shadow_prev_1', 'lower_shadow_prev_1',
               'MACD_prev_2', 'MACD_signal_prev_2', 'MACD_hist_prev_2', 'real_body_prev_2', 
               'is_growing_candle_prev_2', 'upper_shadow_prev_2', 'lower_shadow_prev_2',
               'MACD_prev_3', 'MACD_signal_prev_3', 'MACD_hist_prev_3', 'real_body_prev_3', 
               'is_growing_candle_prev_3', 'upper_shadow_prev_3', 'lower_shadow_prev_3']

In [ ]:
# Масштабирование относительно цены закрытия на минутном таймфрейме
for col in df_result.columns:
    if col not in exclude_futures:
        if col in without_div:
            df_result[col] = (df_result[col] / df_result['close_x'] * 100).round(3)
        else:
            df_result[col] = ((df_result[col] - df_result['close_x']) / df_result['close_x'] * 100).round(3)
# Удаление признака close_x
df_result.drop(columns=['close_x'], 
               inplace=True)
print(f'Количество полных дубликатов: {df_result.duplicated().sum()}')
# Очистка от дубликатов
df_result.drop_duplicates(ignore_index=True,
                          inplace=True)
df_result.info()

In [ ]:
# Удаление записей за субботу и воскресение
df_result = df_result[~((df_result['day_of_week'] == 'Sunday') | (df_result['day_of_week'] == 'Saturday'))]

In [ ]:
df_result['day_of_week'].value_counts()

In [ ]:
# df_result.tail(500).to_csv('./data/500.csv')

Исключаем из датасета признаки, отражающие значимые уровни и не участвующие как технические индикаторы.

In [ ]:
df_result.drop(["long_ext_1.000", "long_ext_1.272", "long_ext_1.414",
                "long_ext_1.618", "long_ext_2.000", "long_ext_2.618",
                "long_ext_3.618", "short_ext_1.000", "short_ext_1.272",
                "short_ext_1.414", "short_ext_1.618", "short_ext_2.000", 
                "short_ext_2.618", "short_ext_3.618"], 
               axis=1, 
               inplace=True)

In [ ]:
df_result.info()

### Обзор и очистка от выбросов

#### Распределение целевых признаков

In [ ]:
# Построение гистограмм распределения целевых признаков
for col in ['predict_to_up', 'predict_to_down', 'predict_level']:
    plot_categorical_feature(df_result, col, col)
for col in ['predict_high', 'predict_low']:
    plot_numerical_feature(df_result, col, col)

#### Оценка распределения признаков и вычисление выбросов.

In [ ]:
normal = []
log_normal = []
unnormal = []
for col in df_result.columns:
    if col not in ['predict_high', 'predict_low', 'predict_to_up', 'predict_to_down', 'predict_level', 'day_of_week']:
        _, p = stats.kstest(df_result[col], 'norm')
        if p > alpha:
            normal.append(col)
        else:
            _, p = stats.kstest(np.log(abs(df_result[col])+1), 'norm')
            if p > alpha:
                log_normal.append(col)
            else:
                unnormal.append(col)
print(f'Количество признаков, распределеных нормально: {len(normal)}')
print(f'Количество признаков, распределеных log-нормально: {len(log_normal)}')
print(f'Количество признаков, c распределением, отличным от нормального: {len(unnormal)}')

Распределение всех числовых признаков отлично от нормального, исходя из этого для очистки от выбросов предпочтительно использовать метод Тьюки.

In [ ]:
def outliers_iqr(data):
    quartile_1, quartile_3 = data.quantile(0.25), data.quantile(0.75),
    iqr = quartile_3 - quartile_1
    lower_bound = quartile_1 - (iqr * 3)
    upper_bound = quartile_3 + (iqr * 3)
    outliers = data[(data < lower_bound) | (data > upper_bound)]
    return outliers.index

Очистке подвергнем признаки с высоким skew, так как очистка от выбросов по всем признакам приводит к сокращению количества записей более, чем на 10%.

Из очистки от выбросов исключаем целевые признаки, так как выбросы в них не имеют существенного влияния при обучении моделей на базе деревьев, а также признаки дней недели торгов и количества минут до пересчета индикаторов.

In [ ]:
outliers_df_index = set()
for col in df_result.columns:
    if col not in ['predict_to_up', 'predict_to_down', 'predict_high', 'predict_low', 'min_to_next',
                   'day_of_week', 'predict_level', 'month_day', 'month', 'hour'] \
        and not pd.api.types.is_bool_dtype(df_result[col]):
        # Правосторонняя асимметрия распределения данных
        if df_result[col].skew() > 15 \
            and df_result[col].min() >= 0:
            outliers = outliers_iqr(np.log1p(df_result[col]))
            outliers_df_index.update(outliers)
        # Левосторонняя асимметрия распределения данных
        elif df_result[col].skew() < -15\
            and df_result[col].max() <= 0:
            outliers = outliers_iqr(np.log1p(-df_result[col]))
            outliers_df_index.update(outliers)
            
df_clean = df_result.drop(index=sorted(outliers_df_index))
print(f"Удалено {len(outliers_df_index)} выбросов.")

In [ ]:
df_clean.shape

Таким образом, выборка df_clean содержит очищенные данные, ввиду особенностей распределений значений признаков, для очистки применялся более мягкий подход, что обеспечило уменьшение количества записей в пределах 6%, что позволяет сохранить репрезентативность выборки.

#### Оценка степени мультиколлинеарности признаков

In [ ]:
# Построение корреляционной матрицы
df_corr = df_clean.corr(method='pearson', numeric_only=True)
# Матрица корреляций
plt.figure(figsize=(15, 15))
sns.heatmap(df_clean.corr(numeric_only=True),
            annot=False, # Подпись данных
            annot_kws = {'size':5},
            vmin=-1, vmax=1,
            center=0,
            cmap='seismic',
            mask=np.triu(df_clean.corr(numeric_only=True)))
plt.show()

Модели на основе деревьев устойчивы к мультиколлинеарности и автоматически выбирают более информативные признаки. Так как точность прогнозов модели представляется более важной по отношению к ее интерпретируемости, стабильности и компактности, то удалять скоррелированные признаки на данном этапе не представляется необходимым.

In [ ]:
# Подсчет количества сильно коррелирующих признаков для каждого признака из пар
dict_strong_corr_features = {}
for i in range(1, df_corr.shape[0]):
    for j in range(0, i):
        if abs(df_corr.iloc[i, j]) >= corr_limit:
            dict_strong_corr_features[df_corr.index[i]] = dict_strong_corr_features.get(df_corr.index[i], 0) + 1
            dict_strong_corr_features[df_corr.columns[j]] = dict_strong_corr_features.get(df_corr.columns[j], 0) + 1
# Сортировка полученных значений
dict_strong_corr_features = dict(sorted(dict_strong_corr_features.items(), key=lambda x: x[1], reverse=True))

In [ ]:
# Сохранение списка сильно коррелирующих факторов для дальнейшего использования
lst_strong_corr_features = list(dict_strong_corr_features.keys())
# Очистка от сильно коррелирующих признаков
df_cleaned_strong_corr = df_clean.copy()
while len(dict_strong_corr_features) > 0:
    # print(next(iter(dict_strong_corr_features)))
    # Удаление наиболее встречающегося сильно скорр. признака
    df_cleaned_strong_corr.drop(next(iter(dict_strong_corr_features)), axis=1, inplace=True)
    # Пересчет матрицы корреляций и вычисление следующего признака для удаления
    df_corr = df_cleaned_strong_corr.corr(method='pearson', numeric_only=True)
    # Подсчет количества сильно коррелирующих признаков для каждого признака из пар
    dict_strong_corr_features.clear()
    for i in range(1, df_corr.shape[0]):
        for j in range(0, i):
            if abs(df_corr.iloc[i, j]) >= corr_limit:
                dict_strong_corr_features[df_corr.index[i]] = dict_strong_corr_features.get(df_corr.index[i], 0) + 1
                dict_strong_corr_features[df_corr.columns[j]] = dict_strong_corr_features.get(df_corr.columns[j], 0) + 1
    # Сортировка полученных значений
    dict_strong_corr_features = dict(sorted(dict_strong_corr_features.items(), key=lambda x: x[1], reverse=True))

# Матрица корреляций
plt.figure(figsize=(15, 15))
sns.heatmap(df_cleaned_strong_corr.corr(numeric_only=True),
            annot=False, # Подпись данных
            annot_kws = {'size':5},
            vmin=-1, vmax=1,
            center=0,
            cmap='seismic',
            mask=np.triu(df_cleaned_strong_corr.corr(numeric_only=True)))
plt.show()

In [ ]:
df_cleaned_strong_corr.shape[1]

In [ ]:
df_cleaned_strong_corr.columns

После удаления сильно скоррелированных признаков датасет содержит 68 признаков, что обеспечит экономию времени на обучение моделей и подбор гиперпараметров.

## Формирование и оценка моделей

In [ ]:
# Выделение целевых признаков
y_high = df_cleaned_strong_corr['predict_high']
y_low = df_cleaned_strong_corr['predict_low']
y_to_up = df_cleaned_strong_corr['predict_to_up']
y_to_down = df_cleaned_strong_corr['predict_to_down']
y_level = df_cleaned_strong_corr['predict_level']
X = df_cleaned_strong_corr.drop(['predict_high', 'predict_low', 'predict_to_up', 'predict_to_down', 'predict_level'], axis=1)

limit_80 = int(X.shape[0]*0.8)

#### Прогнозирование уроня, пересекаемого ценой

In [ ]:
# Разбивка данных на тренировочную и тестовую выборки
X_train_level, X_test_level, y_train_level, y_test_level = train_test_split(X, y_level, test_size=0.2, shuffle=False, random_state=None)
# Размерности выборок
print(f'Размерность обучающей выборки {X_train_level.shape}')
print(f'Размерность тестовой выборки {X_test_level.shape}')
# Рассчет базовых наборов весов
counts = np.bincount(y_train_level)
median = np.median(counts)
base_w = median / counts.astype(float)
scale = 1.2
scaled_weights = (base_w * scale).tolist()
print(scaled_weights)

In [ ]:
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             log_loss, confusion_matrix, roc_auc_score)

# Создание Pool‑ов (оптимальный формат CatBoost) с указанием имен категориальных признаков
train_pool = Pool(X_train_level, y_train_level, cat_features=['day_of_week'])
val_pool = Pool(X_test_level, y_test_level, cat_features=['day_of_week'])

model = CatBoostClassifier(loss_function="MultiClassOneVsAll", # мультиклассовая классификация для несбалансированных классов
                           eval_metric="AUC",            # можно добавить "Accuracy"
                           iterations=1000,
                           learning_rate=0.05,
                           depth=6,
                           l2_leaf_reg=5,
                           class_weights=scaled_weights,
                           random_seed=42,
                           early_stopping_rounds=50)
model.fit(train_pool, eval_set=val_pool, verbose=False)

print(f"Best iteration: {model.best_iteration_}")

#  Предсказания и оценка
pred_proba = model.predict_proba(val_pool)      # shape = (n_samples, K)
pred_class = pred_proba.argmax(axis=1)          # выбираем класс с max‑prob

print("Accuracy:", accuracy_score(y_test_level, pred_class))
print(classification_report(y_test_level, pred_class))

---

In [ ]:
# Пространство поиска гиперпараметров
space_level = {"depth": hp.quniform("depth", 4, 10, 1),
               "learning_rate": hp.loguniform("learning_rate", np.log(0.01), np.log(0.2)),
               "l2_leaf_reg": hp.uniform("l2_leaf_reg", 1.0, 10.0),
               "border_count": hp.qloguniform("border_count", np.log(64), np.log(256), 1),
               "bagging_temperature": hp.uniform("bagging_temperature", 0.0, 1.0),
               "class_weights_scale": hp.uniform("class_weights_scale", 0.5, 2.0),
               "iterations": hp.choice("iterations", [500, 800, 1200])}

In [ ]:
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from sklearn.metrics import precision_score, log_loss


def hyperopt_rfc_level(params, X=X_train_level, y=y_train_level, random_state=rs):
    params["depth"] = int(params["depth"])
    params["border_count"] = int(params["border_count"])
    scale = params.pop("class_weights_scale")
    scaled_weights = (base_w * scale).tolist()
    fixed_params = {"loss_function": "MultiClassOneVsAll",
                    "eval_metric": "AUC",
                    "verbose": False,
                    "early_stopping_rounds": 50,
                    "use_best_model": True,
                    "class_weights": scaled_weights}
    fixed_params.update(params)
    model = CatBoostClassifier(**fixed_params)
    model.fit(train_pool, eval_set=val_pool, verbose=False)
    
    val_pred = model.predict(val_pool)
    macro_prec = precision_score(y_test_level,
                                 val_pred,
                                 average="macro",
                                 zero_division=0)
    loss = 1.0 - macro_prec
    
    prob = model.predict_proba(val_pool)
    aux_logloss = log_loss(y_test_level, prob)

    return {"loss": loss,
            "status": STATUS_OK,
            "metrics": {"macro_precision": macro_prec,
                        "logloss": aux_logloss,},
            "params": fixed_params}

In [ ]:
# Подбор гиперпараметров
trials_level = Trials() # используется для логирования результатов

if recalculating_model_level:
    best_level=fmin(hyperopt_rfc_level, # функция 
                    space=space_level, # пространство гиперпараметров
                    algo=tpe.suggest, # алгоритм оптимизации, установлен по умолчанию, задавать необязательно
                    max_evals=10, # максимальное количество итераций
                    trials=trials_level, # логирование результатов
                    rstate=np.random.default_rng(rs))# фиксируем для повторяемости результата

In [ ]:
print("\n=== Best hyper‑parameters found ===")
# Convert the integer‑typed entries back to int for readability
best_level["depth"] = int(best_level["depth"])
best_level["border_count"] = int(best_level["border_count"])
print(best_level)

# If you want to inspect the trial with the highest macro‑precision:
best_trial = max(trials_level.trials, key=lambda t: t["result"]["metrics"]["macro_precision"])
print("\nBest trial metrics:")
print(best_trial["result"]["metrics"])
print("\nCorresponding parameters (including class_weights):")
print(best_trial["result"]["params"])

In [ ]:
# Вывод результатов подбора
if recalculating_model_level:
    best_trial_level = best_trial
    model_level_hyp_res = best_trial_level['result']['model']
    print("\n=== Параметры ===")
    print(best_trial_level['result']['params'])
    print("\n=== Средний CV-score (AUC) ===")
    print(-best_trial_level['result']['loss'])
else:
    params_level = {'colsample_bytree': 0.9564157614482433, 
                    'eval_metric': 'auc', 
                    'gamma': 0.0008005184312606462, 
                    'learning_rate': 0.11914558625685459, 
                    'max_depth': 10, 
                    'min_child_weight': 1.3797809963728616, 
                    'n_estimators': 750, 
                    'n_jobs': -1, 
                    'num_class': 15, 
                    'objective': 'multi:softprob', 
                    'reg_alpha': 0.3077617464313159, 
                    'reg_lambda': 0.9561759020734177, 
                    'seed': 42, 
                    'subsample': 0.5321434316547746, 
                    'tree_method': 'hist'}

    print('Параметры модели не пересчитывались\nСредний CV-score (F1) 0.29238028438706315')

Подбор гиперпараметров осуществлен при пороге для определения направления движения цены в процентах
level_price_movement_direction = 0.05:  
100%|██████████| 7/7 [5:41:17<00:00, 2925.30s/trial, best loss: -0.29238028438706315]  
Наилучшие значения гиперпараметров {'bootstrap': True, 'max_depth': 17.0, 'max_features': 0.586364059503943, 'min_samples_leaf': 7.0, 'min_samples_split': 13.0, 'model_type': 'rf', 'n_estimators': 80.0}  
Для перезапуска подбора гиперпараметров изменить recalculating_model_to_up на True.

100%|██████████| 10/10 [1:59:45<00:00, 718.59s/trial, best loss: -0.9015274335619571]
=== Параметры ===
{'colsample_bytree': 0.6289739944343503, 'eval_metric': 'auc', 'gamma': 0.28692416723333913, 'learning_rate': 0.18014068735873387, 'max_depth': 7, 'min_child_weight': 1.337115273793821, 'n_estimators': 1850, 'n_jobs': -1, 'objective': 'binary:logistic', 'random_state': 42, 'reg_alpha': 0.03223527058046773, 'reg_lambda': 9.494904155057856e-05, 'scale_pos_weight': 1.049586464931545, 'subsample': 0.9018909441059156}

100%|██████████| 10/10 [5:44:24<00:00, 2066.44s/trial, best loss: -0.9898714859370575]
=== Параметры ===
{'colsample_bytree': 0.9564157614482433, 'eval_metric': 'auc', 'gamma': 0.0008005184312606462, 'learning_rate': 0.11914558625685459, 'max_depth': 10, 'min_child_weight': 1.3797809963728616, 'n_estimators': 750, 'n_jobs': -1, 'num_class': 15, 'objective': 'multi:softprob', 'reg_alpha': 0.3077617464313159, 'reg_lambda': 0.9561759020734177, 'seed': 42, 'subsample': 0.5321434316547746, 'tree_method': 'hist'}
=== Средний CV-score (AUC) ===
0.9898714859370575

In [ ]:
# Обучение модели и получение ее оценки
if not recalculating_model_level:
    model_level_hyp_res = XGBClassifier(**params_level)
model_level_hyp_res.fit(X_train_level, y_train_level)
# y_pred_train_level = model_level_hyp_res.predict(X_train_level)
# y_pred_test_level = model_level_hyp_res.predict(X_test_level)
# Получение метрик модели
# print('-------На тренировочных данных-------')
# print(f'Точность: {metrics.precision_score(y_train_level, y_pred_train_level):.4f}    Полнота: {metrics.recall_score(y_train_level, y_pred_train_level):.4f}')
# print('-------На тестовых данных------------')
# print(f'Точность: {metrics.precision_score(y_test_level, y_pred_test_level):.4f}    Полнота: {metrics.recall_score(y_test_level, y_pred_test_level):.4f}')

# 5. Оценка на тесте
y_pred  = model_level_hyp_res.predict(X_test_level)
y_proba = model_level_hyp_res.predict_proba(X_test_level)

acc  = accuracy_score(y_test_level, y_pred)
logl = log_loss(y_test_level, y_proba)
f1   = f1_score(y_test_level, y_pred, average='macro')

print(f'Accuracy : {acc:.4f}')
print(f'Log-loss : {logl:.4f}')
print(f'Macro-F1 : {f1:.4f}')
print('\nClassification report')
print(classification_report(y_test_level, y_pred, target_names=list(dict_levels.keys())))

# Confusion matrix
cm = confusion_matrix(y_test_level, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=list(dict_levels.keys()), yticklabels=list(dict_levels.keys()))
plt.title('Confusion matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# ROC‑AUC (OvR) – если >2 класса
if len(list(dict_levels.keys())) > 2:
    auc = roc_auc_score(y_test_level, y_proba, multi_class='ovr')
    print(f'ROC-AUC (OvR): {auc:.4f}')

# -------------------------------------------------
# 6. Важность признаков
# -------------------------------------------------
xgb.plot_importance(model_level_hyp_res, max_num_features=30, importance_type='gain',
                    title='Top 30 features')
plt.show()

---

#### Прогнозирование направления движения цены вверх

In [ ]:
# Разбивка данных на тренировочную и тестовую выборки
X_train_up, X_test_up, y_train_up, y_test_up = train_test_split(X, y_to_up, test_size=0.2, shuffle=False, random_state=None)
# Размерности выборок
print(f'Размерность обучающей выборки {X_train_up.shape}')
print(f'Размерность тестовой выборки {X_test_up.shape}')

In [ ]:
# Создание Pool‑ов (оптимальный формат CatBoost) с указанием имен категориальных признаков
train_pool = Pool(X_train_up, y_train_up, cat_features=['day_of_week'])
val_pool = Pool(X_test_up, y_test_up, cat_features=['day_of_week'])

cnt = np.bincount(y_train_up)
median_cnt = np.median(cnt)
class_weights = {0: median_cnt / cnt[0], 1: median_cnt / cnt[1]}

model_to_up = CatBoostClassifier(eval_metric='Precision',
                                 class_weights=class_weights)
model_to_up.fit(train_pool, eval_set=val_pool, verbose=False)

# Оценка
pred_proba_train = model_to_up.predict_proba(train_pool)[:, 1]   # вероятность класса 1
pred_label_train = (pred_proba_train >= 0.5).astype(int)
pred_proba_test = model_to_up.predict_proba(val_pool)[:, 1]   # вероятность класса 1
pred_label_test = (pred_proba_test >= 0.5).astype(int)

print('TRAIN:')
print('AUC :', roc_auc_score(y_train_up, pred_proba_train))
print('Acc :', accuracy_score(y_train_up, pred_label_train))
print('TEST:')
print('AUC :', roc_auc_score(y_test_up, pred_proba_test))
print('Acc :', accuracy_score(y_test_up, pred_label_test))
#---------------------------------
print('-------На тренировочных данных-------')
print(f'Точность: {metrics.precision_score(y_train_up, pred_label_train):.4f}    Полнота: {metrics.recall_score(y_train_up, pred_label_train):.4f}')
print('-------На тестовых данных------------')
print(f'Точность: {metrics.precision_score(y_test_up, pred_label_test):.4f}    Полнота: {metrics.recall_score(y_test_up, pred_label_test):.4f}')

In [ ]:
y_pred_train_up = model_to_up.predict(X_train_up)
y_pred_test_up = model_to_up.predict(X_test_up)
# Получение метрик модели
print('-------На тренировочных данных-------')
print(f'Точность: {metrics.precision_score(y_train_up, y_pred_train_up):.4f}    Полнота: {metrics.recall_score(y_train_up, y_pred_train_up):.4f}')
print('-------На тестовых данных------------')
print(f'Точность: {metrics.precision_score(y_test_up, y_pred_test_up):.4f}    Полнота: {metrics.recall_score(y_test_up, y_pred_test_up):.4f}')

#### Подбор гиперпараметров модели "Прогнозирование направления движения цены вверх"

In [ ]:
# Пространство поиска гиперпараметров
space_up = {"learning_rate": hp.loguniform("learning_rate", np.log(0.01), np.log(0.3)),
            "n_estimators": hp.quniform("n_estimators", 200, 2000, 50),
            "max_depth": hp.quniform("max_depth", 3, 12, 1),
            "min_child_weight": hp.loguniform("min_child_weight", np.log(0.1), np.log(10)),
            "subsample": hp.uniform("subsample", 0.6, 1.0),
            "colsample_bytree": hp.uniform("colsample_bytree", 0.6, 1.0),
            "gamma": hp.loguniform("gamma", np.log(1e-8), np.log(5.0)),
            "reg_alpha": hp.loguniform("reg_alpha", np.log(1e-8), np.log(10.0)),
            "reg_lambda": hp.loguniform("reg_lambda", np.log(1e-8), np.log(10.0)),
            # Балансировка классов – широкий диапазон
            "scale_pos_weight": hp.loguniform("scale_pos_weight", np.log(0.5), np.log(50.0)),
            "objective": "binary:logistic",
            "eval_metric": "auc",
            "random_state": rs,
            "n_jobs": -1}

In [ ]:
def hyperopt_rfc_up(params, cv=5, X=X_train_up, y=y_train_up, random_state=rs):
    params["n_estimators"] = int(params["n_estimators"])
    params["max_depth"]    = int(params["max_depth"])

    model = XGBClassifier(**params)
    
    # --- стратифицированная K‑fold кросс‑валидация -------------------------
    cvs = StratifiedKFold(n_splits=cv, shuffle=True, random_state=random_state)
    scores = cross_val_score(
        model, X, y,
        cv=cvs,
        scoring='f1',      # или 'accuracy', 'f1', в зависимости от задачи
        n_jobs=-1)
    mean_score = scores.mean()
    
    # --- возврат результата ------------------------------------------------
    return {'loss': -mean_score,
            'status': STATUS_OK,
            'model': model,
            'params': params}

In [ ]:
# Подбор гиперпараметров
trials_up = Trials() # используется для логирования результатов

if recalculating_model_to_up:
    best_up=fmin(hyperopt_rfc_up, # функция 
                 space=space_up, # пространство гиперпараметров
                 algo=tpe.suggest, # алгоритм оптимизации, установлен по умолчанию, задавать необязательно
                 max_evals=10, # максимальное количество итераций
                 trials=trials_up, # логирование результатов
                 rstate=np.random.default_rng(rs))# фиксируем для повторяемости результата

In [ ]:
# Вывод результатов подбора
if recalculating_model_to_up:
    best_trial_up = trials_up.best_trial
    model_to_up_hyp_res = best_trial_up['result']['model']
    print("\n=== Параметры ===")
    print(best_trial_up['result']['params'])
    print("\n=== Средний CV-score (F1) ===")
    print(-best_trial_up['result']['loss'])
else:
    params_up = {'colsample_bytree': 0.6289739944343503, 
                 'eval_metric': 'auc', 
                 'gamma': 0.28692416723333913, 
                 'learning_rate': 0.18014068735873387, 
                 'max_depth': 7, 
                 'min_child_weight': 1.337115273793821, 
                 'n_estimators': 1850, 
                 'n_jobs': -1, 
                 'objective': 'binary:logistic', 
                 'random_state': 42, 
                 'reg_alpha': 0.03223527058046773, 
                 'reg_lambda': 9.494904155057856e-05, 
                 'scale_pos_weight': 1.049586464931545, 
                 'subsample': 0.9018909441059156}

    print('Параметры модели не пересчитывались\nСредний CV-score (F1) 0.29238028438706315')

Подбор гиперпараметров осуществлен при пороге для определения направления движения цены в процентах
level_price_movement_direction = 0.05:  
100%|██████████| 7/7 [5:41:17<00:00, 2925.30s/trial, best loss: -0.29238028438706315]  
Наилучшие значения гиперпараметров {'bootstrap': True, 'max_depth': 17.0, 'max_features': 0.586364059503943, 'min_samples_leaf': 7.0, 'min_samples_split': 13.0, 'model_type': 'rf', 'n_estimators': 80.0}  
Для перезапуска подбора гиперпараметров изменить recalculating_model_to_up на True.


100%|██████████| 10/10 [1:59:45<00:00, 718.59s/trial, best loss: -0.9015274335619571]
=== Параметры ===
{'colsample_bytree': 0.6289739944343503, 'eval_metric': 'auc', 'gamma': 0.28692416723333913, 'learning_rate': 0.18014068735873387, 'max_depth': 7, 'min_child_weight': 1.337115273793821, 'n_estimators': 1850, 'n_jobs': -1, 'objective': 'binary:logistic', 'random_state': 42, 'reg_alpha': 0.03223527058046773, 'reg_lambda': 9.494904155057856e-05, 'scale_pos_weight': 1.049586464931545, 'subsample': 0.9018909441059156}

100%|██████████| 10/10 [1:25:41<00:00, 514.18s/trial, best loss: -0.837903687345869]
=== Параметры ===
{'colsample_bytree': 0.6289739944343503, 'eval_metric': 'auc', 'gamma': 0.28692416723333913, 'learning_rate': 0.18014068735873387, 'max_depth': 7, 'min_child_weight': 1.337115273793821, 'n_estimators': 1850, 'n_jobs': -1, 'objective': 'binary:logistic', 'random_state': 42, 'reg_alpha': 0.03223527058046773, 'reg_lambda': 9.494904155057856e-05, 'scale_pos_weight': 1.049586464931545, 'subsample': 0.9018909441059156}
=== Средний CV-score (F1) ===
0.837903687345869

In [ ]:
# Обучение модели и получение ее оценки
if not recalculating_model_to_up:
    model_to_up_hyp_res = XGBClassifier(**params_up)
model_to_up_hyp_res.fit(X_train_up, y_train_up)
y_pred_train_up = model_to_up_hyp_res.predict(X_train_up)
y_pred_test_up = model_to_up_hyp_res.predict(X_test_up)
# Получение метрик модели
print('-------На тренировочных данных-------')
print(f'Точность: {metrics.precision_score(y_train_up, y_pred_train_up):.4f}    Полнота: {metrics.recall_score(y_train_up, y_pred_train_up):.4f}')
print('-------На тестовых данных------------')
print(f'Точность: {metrics.precision_score(y_test_up, y_pred_test_up):.4f}    Полнота: {metrics.recall_score(y_test_up, y_pred_test_up):.4f}')

#### Прогнозирование направления движения цены вниз

In [ ]:
# # Разбивка данных на тренировочную и тестовую выборки
# X_train_down, X_test_down, y_train_down, y_test_down = train_test_split(X, y_to_down, test_size=0.3, stratify=y_to_down, random_state=rs)
# # Размерности выборок
# print(f'Размерность обучающей выборки {X_train_down.shape}')
# print(f'Размерность тестовой выборки {X_test_down.shape}')

In [ ]:
# Разбивка данных на тренировочную и тестовую выборки
X_train_down, X_test_down, y_train_down, y_test_down = train_test_split(X, y_to_down, test_size=0.2, shuffle=False, random_state=None)
# Размерности выборок
print(f'Размерность обучающей выборки {X_train_down.shape}')
print(f'Размерность тестовой выборки {X_test_down.shape}')

In [ ]:
# Создаем модель случайного леса из 100 деревьев с максимальной глубиной 20 для каждого дерева 
model_to_down = RandomForestClassifier(n_estimators=100,    # начнем со 100 деревьев
                                     max_depth=10,        # ограничим глубину
                                     max_features='sqrt', # ускоряет, предотвращает переобучение
                                     min_samples_leaf=5,  # делает деревья устойчивее
                                     n_jobs=-1,           # используем все CPU
                                     random_state=rs)     # воспроизводимость
# Обучаем модель
model_to_down.fit(X_train_down, y_train_down)

In [ ]:
y_pred_train_down = model_to_down.predict(X_train_down)
y_pred_test_down = model_to_down.predict(X_test_down)
# Получение метрик модели
print('-------На тренировочных данных-------')
print(f'Точность: {metrics.precision_score(y_train_down, y_pred_train_down):.4f}    Полнота: {metrics.recall_score(y_train_down, y_pred_train_down):.4f}')
print('-------На тестовых данных------------')
print(f'Точность: {metrics.precision_score(y_test_down, y_pred_test_down):.4f}    Полнота: {metrics.recall_score(y_test_down, y_pred_test_down):.4f}')

In [ ]:
# Создаем модель случайного леса из 100 деревьев с максимальной глубиной 20 для каждого дерева 
model_to_down = XGBClassifier(n_estimators=100,    # начнем со 100 деревьев
                            max_depth=10,        # ограничим глубину
                            random_state=rs)
# Обучаем модель
model_to_down.fit(X_train_down, y_train_down)

In [ ]:
y_pred_train_down = model_to_down.predict(X_train_down)
y_pred_test_down = model_to_down.predict(X_test_down)
# Получение метрик модели
print('-------На тренировочных данных-------')
print(f'Точность: {metrics.precision_score(y_train_down, y_pred_train_down):.4f}    Полнота: {metrics.recall_score(y_train_down, y_pred_train_down):.4f}')
print('-------На тестовых данных------------')
print(f'Точность: {metrics.precision_score(y_test_down, y_pred_test_down):.4f}    Полнота: {metrics.recall_score(y_test_down, y_pred_test_down):.4f}')

#### Подбор гиперпараметров модели "Прогнозирование направления движения цены вниз"

In [ ]:
# Пространство поиска гиперпараметров
space_down = {"learning_rate": hp.loguniform("learning_rate", np.log(0.01), np.log(0.3)),
            "n_estimators": hp.quniform("n_estimators", 200, 2000, 50),
            "max_depth": hp.quniform("max_depth", 3, 12, 1),
            "min_child_weight": hp.loguniform("min_child_weight", np.log(0.1), np.log(10)),
            "subsample": hp.uniform("subsample", 0.6, 1.0),
            "colsample_bytree": hp.uniform("colsample_bytree", 0.6, 1.0),
            "gamma": hp.loguniform("gamma", np.log(1e-8), np.log(5.0)),
            "reg_alpha": hp.loguniform("reg_alpha", np.log(1e-8), np.log(10.0)),
            "reg_lambda": hp.loguniform("reg_lambda", np.log(1e-8), np.log(10.0)),
            # Балансировка классов – широкий диапазон
            "scale_pos_weight": hp.loguniform("scale_pos_weight", np.log(0.5), np.log(50.0)),
            "objective": "binary:logistic",
            "eval_metric": "auc",
            "random_state": rs,
            "n_jobs": -1}

In [ ]:
def hyperopt_rfc_down(params, cv=5, X=X_train_down, y=y_train_down, random_state=rs):
    params["n_estimators"] = int(params["n_estimators"])
    params["max_depth"]    = int(params["max_depth"])

    model = XGBClassifier(**params)
    
    # --- стратифицированная K‑fold кросс‑валидация -------------------------
    cvs = StratifiedKFold(n_splits=cv, shuffle=True, random_state=random_state)
    scores = cross_val_score(
        model, X, y,
        cv=cvs,
        scoring='f1',      # или 'accuracy', 'f1', в зависимости от задачи
        n_jobs=-1)
    mean_score = scores.mean()
    
    # --- возврат результата ------------------------------------------------
    return {'loss': -mean_score,
            'status': STATUS_OK,
            'model': model,
            'params': params}

In [ ]:
# Подбор гиперпараметров
trials_down = Trials() # используется для логирования результатов

if recalculating_model_to_down:
    best_down=fmin(hyperopt_rfc_down, # функция 
                 space=space_down, # пространство гиперпараметров
                 algo=tpe.suggest, # алгоритм оптимизации, установлен по умолчанию, задавать необязательно
                 max_evals=10, # максимальное количество итераций
                 trials=trials_down, # логирование результатов
                 rstate=np.random.default_rng(rs))# фиксируем для повторяемости результата

In [ ]:
# Вывод результатов подбора
if recalculating_model_to_down:
    best_trial_down = trials_down.best_trial
    model_to_down_hyp_res = best_trial_down['result']['model']
    print("\n=== Параметры ===")
    print(best_trial_down['result']['params'])
    print("\n=== Средний CV-score (F1) ===")
    print(-best_trial_down['result']['loss'])
else:
    params_down = {'colsample_bytree': 0.6289739944343503, 
                   'eval_metric': 'auc', 
                   'gamma': 0.28692416723333913, 
                   'learning_rate': 0.18014068735873387, 
                   'max_depth': 7, 
                   'min_child_weight': 1.337115273793821, 
                   'n_estimators': 1850, 
                   'n_jobs': -1, 
                   'objective': 'binary:logistic', 
                   'random_state': 42, 
                   'reg_alpha': 0.03223527058046773, 
                   'reg_lambda': 9.494904155057856e-05, 
                   'scale_pos_weight': 1.049586464931545, 
                   'subsample': 0.9018909441059156}

    print('Параметры модели не пересчитывались\nСредний CV-score (F1) 0.29238028438706315')

Подбор гиперпараметров осуществлен при пороге для определения направления движения цены в процентах
level_price_movement_direction = 0.15:  
100%|██████████| 7/7 [5:51:42<00:00, 3014.71s/trial, best loss: -0.3643594452798749]  
Наилучшие значения гиперпараметров {'bootstrap': True, 'max_depth': 17.0, 'max_features': 0.586364059503943, 'min_samples_leaf': 7.0, 'min_samples_split': 13.0, 'model_type': 'rf', 'n_estimators': 80.0}  
Для перезапуска подбора гиперпараметров изменить recalculating_model_to_down на True.

100%|██████████| 10/10 [1:24:33<00:00, 507.36s/trial, best loss: -0.8348661581987926]
=== Параметры ===
{'colsample_bytree': 0.6289739944343503, 'eval_metric': 'auc', 'gamma': 0.28692416723333913, 'learning_rate': 0.18014068735873387, 'max_depth': 7, 'min_child_weight': 1.337115273793821, 'n_estimators': 1850, 'n_jobs': -1, 'objective': 'binary:logistic', 'random_state': 42, 'reg_alpha': 0.03223527058046773, 'reg_lambda': 9.494904155057856e-05, 'scale_pos_weight': 1.049586464931545, 'subsample': 0.9018909441059156}
=== Средний CV-score (F1) ===
0.8348661581987926

In [ ]:
# Обучение модели и получение ее оценки
if not recalculating_model_to_down:
    model_to_down_hyp_res = XGBClassifier(**params_down)
model_to_down_hyp_res.fit(X_train_down, y_train_down)
y_pred_train_down = model_to_down_hyp_res.predict(X_train_down)
y_pred_test_down = model_to_down_hyp_res.predict(X_test_down)
# Получение метрик модели
print('-------На тренировочных данных-------')
print(f'Точность: {metrics.precision_score(y_train_down, y_pred_train_down):.4f}    Полнота: {metrics.recall_score(y_train_down, y_pred_train_down):.4f}')
print('-------На тестовых данных------------')
print(f'Точность: {metrics.precision_score(y_test_down, y_pred_test_down):.4f}    Полнота: {metrics.recall_score(y_test_down, y_pred_test_down):.4f}')

In [ ]:
plt_up = y_to_up.astype(int)
plt_down = -y_to_down.astype(int)
plt_pred_up = model_to_up_hyp_res.predict(X).astype(int)
plt_pred_down = -model_to_down_hyp_res.predict(X).astype(int)
#строим график
len_plot = 1000
fig = go.Figure()
# Построение коробчатой диаграммы
fig.add_trace(go.Scatter(x=df_cleaned_strong_corr.tail(len_plot).index,
                                 y=plt_up.tail(len_plot),
                                 mode = 'markers',
                                 marker=dict( size=3, color='green', symbol='square'),
                                 fill = 'tozeroy',
                                 showlegend=True,
                                 name='true_UP')) # Название набора данных
fig.add_trace(go.Scatter(x=df_cleaned_strong_corr.tail(len_plot).index,
                                 y=plt_down.tail(len_plot),
                                 mode = 'markers',
                                 marker=dict( size=3, color='red', symbol='square'),
                                 fill = 'tozeroy',
                                 showlegend=True,
                                 name='true_DOWN')) # Название набора данных
fig.add_trace(go.Scatter(x=df_cleaned_strong_corr.tail(len_plot).index,
                                 y=plt_pred_up[:-len_plot],
                                 mode = 'markers',
                                 marker=dict( size=3, color='blue', symbol='square'),
                                 fill = 'tozeroy',
                                 showlegend=True,
                                 name='predict UP')) # Название набора данных
fig.add_trace(go.Scatter(x=df_cleaned_strong_corr.tail(len_plot).index,
                                 y=plt_pred_down[:-len_plot],
                                 mode = 'markers',
                                 marker=dict( size=3, color='blue', symbol='square'),
                                 fill = 'tozeroy',
                                 showlegend=True,
                                 name='predict DOWN')) # Название набора данных

fig.update_layout(autosize = False, width = 1500, height = 620, # Размер полотна
                             title_text=f'Оценка распределения движения цены <br>(последние {len_plot})',
                             title_x=0.5,
                             xaxis_title='Последовательность получения данных',
                             yaxis_title='Сигнал')
fig.show()

#### Объединение и интерпретация результатов моделей

Высчитать полноту предсказаний моделей движения цены вверх/вниз

В случае построения стратегии на основании получения сигналов от обученных моделей по определению направления движения цены точность данных сигналов составляет оценочно 97.5%, достоверно охватывает порядка 23% всех движений, что составляет 12% от всех значений цен. То есть, из всего диапазона получаемых цен около 49% имеют движение (вверх или вниз на 0.15% от цены закрытия), данные модели позволяют точно идентифицировать из них порядка 12%.  
На данном этапе это удовлетворительный результат для построения тестовой рабочей стратегии.

---

#### Прогнозирование максимальной цены за период

In [ ]:
# # Разбивка данных на тренировочную и тестовую выборки
# X_train_high, X_test_high, y_train_high, y_test_high = train_test_split(X, y_high, test_size=0.3, random_state=rs)
# # Размерности выборок
# print(f'Размерность обучающей выборки {X_train_high.shape}')
# print(f'Размерность тестовой выборки {X_test_high.shape}')

In [ ]:
# Разбивка данных на тренировочную и тестовую выборки
X_train_high, X_test_high, y_train_high, y_test_high = train_test_split(X, y_high, test_size=0.2, shuffle=False, random_state=None)
# Размерности выборок
print(f'Размерность обучающей выборки {X_train_high.shape}')
print(f'Размерность тестовой выборки {X_test_high.shape}')

In [ ]:
# Создаем модель случайного леса из 100 деревьев с максимальной глубиной 20 для каждого дерева 
model_high = RandomForestRegressor(n_estimators=100,    # начнем со 100 деревьев
                                   max_depth=10,        # ограничим глубину
                                   max_features='sqrt', # ускоряет, предотвращает переобучение
                                   min_samples_leaf=5,  # делает деревья устойчивее
                                   n_jobs=-1,           # используем все CPU
                                   random_state=rs)     # воспроизводимость
# Обучаем модель
model_high.fit(X_train_high, y_train_high)

In [ ]:
y_pred_train_high = model_high.predict(X_train_high)
y_pred_test_high = model_high.predict(X_test_high)
# Получение метрик модели
print('-------На тренировочных данных-------')
print(f'MAE: {metrics.mean_absolute_error(y_train_high, y_pred_train_high):.4f}    MSE: {metrics.mean_squared_error(y_train_high, y_pred_train_high):.4f}')
print('-------На тестовых данных------------')
print(f'MAE: {metrics.mean_absolute_error(y_test_high, y_pred_test_high):.4f}    MSE: {metrics.mean_squared_error(y_test_high, y_pred_test_high):.4f}')

#### Подбор гиперпараметров модели "Прогнозирование максимальной цены за период"

In [ ]:
# зададим пространство поиска гиперпараметров
space_high={"learning_rate": hp.loguniform("learning_rate", np.log(0.01), np.log(0.3)),
            "n_estimators": hp.quniform("n_estimators", 200, 2500, 50),
            "max_depth": hp.quniform("max_depth", 3, 12, 1),
            "min_child_weight": hp.loguniform("min_child_weight", np.log(0.1), np.log(10)),
            "subsample": hp.uniform("subsample", 0.6, 1.0),
            "colsample_bytree": hp.uniform("colsample_bytree", 0.6, 1.0),
            "gamma": hp.loguniform("gamma", np.log(1e-8), np.log(5.0)),
            "reg_alpha": hp.loguniform("reg_alpha", np.log(1e-8), np.log(10.0)),
            "reg_lambda": hp.loguniform("reg_lambda", np.log(1e-8), np.log(10.0)),
            "objective": hp.choice("objective", ["reg:squarederror", "reg:pseudohubererror"]),
            "eval_metric": hp.choice("eval_metric", ["rmse", "mae"]),
            "random_state": 42,
            "n_jobs": -1,
            "verbosity": 0}

In [ ]:
def hyperopt_rfr_high(params, cv=3, X=X_train_high, y=y_train_high, random_state=rs):
    params["n_estimators"] = int(params["n_estimators"])
    params["max_depth"]    = int(params["max_depth"])

    model = XGBRegressor(**params)
    
    scores = cross_val_score(model, X, y, cv=cv, scoring='neg_mean_squared_error', n_jobs=-1)
    mean_score = scores.mean()
    # --- возврат результата ------------------------------------------------
    return {'loss': -mean_score,
            'status': STATUS_OK,
            'model': model,
            'params': params}

In [ ]:
# Подбор гиперпараметров
trials_high = Trials() # используется для логирования результатов

if recalculating_model_high:
    best_down=fmin(hyperopt_rfr_high, # функция 
                   space=space_high, # пространство гиперпараметров
                   algo=tpe.suggest, # алгоритм оптимизации, установлен по умолчанию, задавать необязательно
                   max_evals=15, # максимальное количество итераций
                   trials=trials_high, # логирование результатов
                   rstate=np.random.default_rng(rs))# фиксируем для повторяемости результата

In [ ]:
# Вывод результатов подбора
if recalculating_model_high:
    best_trial_high = trials_high.best_trial
    model_high_hyp_res = best_trial_high['result']['model']
    print("\n=== Параметры ===")
    print(best_trial_high['result']['params'])
    print("\n=== Средний CV-score (MSE) ===")
    print(-best_trial_high['result']['loss'])
else:
    params_high = {}
    print('Параметры модели не пересчитывались\nСредний CV-score (MSE) 0.025091218820831842')

Подбор гиперпараметров осуществлен при пороге для определения направления движения цены в процентах
level_price_movement_direction = 0.15:  
100%|██████████| 7/7 [6:22:48<00:00, 3281.14s/trial, best loss: 0.1310551597934567]
Наилучшие значения гиперпараметров {'bootstrap': False, 'max_depth': 16.0, 'max_features': 0.7562196798242258, 'min_samples_leaf': 9.0, 'model_type': 'rf', 'n_estimators': 330.0}  
Для перезапуска подбора гиперпараметров изменить recalculating_model_high на True.

100%|██████████| 15/15 [13:11:53<00:00, 3167.57s/trial, best loss: 0.025091218820831842]
{'bootstrap': False, 'max_depth': 20.0, 'max_features': 0.7014226785922257, 'min_samples_leaf': 1.0, 'model_type': 'rf', 'n_estimators': 400.0}

100%|██████████| 15/15 [1:34:15<00:00, 377.00s/trial, best loss: 0.002597137431924542]
=== Параметры ===
{'colsample_bytree': 0.8422597946469974, 'eval_metric': 'mae', 'gamma': 0.00026731209746217214, 'learning_rate': 0.032452172710781914, 'max_depth': 11, 'min_child_weight': 1.2743849764010975, 'n_estimators': 2400, 'n_jobs': -1, 'objective': 'reg:squarederror', 'random_state': 42, 'reg_alpha': 2.5996973016007112e-06, 'reg_lambda': 2.3514904339337348e-08, 'subsample': 0.6138813743427279, 'verbosity': 0}
=== Средний CV-score (MSE) ===
-0.002597137431924542

In [ ]:
# Обучение модели и получение ее оценки
if not recalculating_model_high:
    model_high_hyp_res = GradientBoostingRegressor(**params_high)
model_high_hyp_res.fit(X_train_high, y_train_high)
y_pred_train_high = model_high_hyp_res.predict(X_train_high)
y_pred_test_high = model_high_hyp_res.predict(X_test_high)
# Получение метрик модели
print('-------На тренировочных данных-------')
print(f'MAE: {metrics.mean_absolute_error(y_train_high, y_pred_train_high):.4f}    MSE: {metrics.mean_squared_error(y_train_high, y_pred_train_high):.4f}')
print('-------На тестовых данных------------')
print(f'MAE: {metrics.mean_absolute_error(y_test_high, y_pred_test_high):.4f}    MSE: {metrics.mean_squared_error(y_test_high, y_pred_test_high):.4f}')

#### Прогнозирование минимальной цены за период

In [ ]:
# # Разбивка данных на тренировочную и тестовую выборки
# X_train_low, X_test_low, y_train_low, y_test_low = train_test_split(X, y_low, test_size=0.3, random_state=rs)
# # Размерности выборок
# print(f'Размерность обучающей выборки {X_train_low.shape}')
# print(f'Размерность тестовой выборки {X_test_low.shape}')

In [ ]:
# Разбивка данных на тренировочную и тестовую выборки
X_train_low, X_test_low, y_train_low, y_test_low = train_test_split(X, y_low, test_size=0.2, shuffle=False, random_state=None)
# Размерности выборок
print(f'Размерность обучающей выборки {X_train_low.shape}')
print(f'Размерность тестовой выборки {X_test_low.shape}')

In [ ]:
# Создаем модель случайного леса из 100 деревьев с максимальной глубиной 20 для каждого дерева 
model_low = RandomForestRegressor(n_estimators=100,    # начнем со 100 деревьев
                                   max_depth=10,        # ограничим глубину
                                   max_features='sqrt', # ускоряет, предотвращает переобучение
                                   min_samples_leaf=5,  # делает деревья устойчивее
                                   n_jobs=-1,           # используем все CPU
                                   random_state=rs)     # воспроизводимость
# Обучаем модель
model_low.fit(X_train_low, y_train_low)

In [ ]:
y_pred_train_low = model_low.predict(X_train_low)
y_pred_test_low = model_low.predict(X_test_low)
# Получение метрик модели
print('-------На тренировочных данных-------')
print(f'MAE: {metrics.mean_absolute_error(y_train_low, y_pred_train_low):.4f}    MSE: {metrics.mean_squared_error(y_train_low, y_pred_train_low):.4f}')
print('-------На тестовых данных------------')
print(f'MAE: {metrics.mean_absolute_error(y_test_low, y_pred_test_low):.4f}    MSE: {metrics.mean_squared_error(y_test_low, y_pred_test_low):.4f}')

#### Подбор гиперпараметров модели "Прогнозирование минимальной цены за период"

In [ ]:
# зададим пространство поиска гиперпараметров
space_low={"learning_rate": hp.loguniform("learning_rate", np.log(0.01), np.log(0.3)),
            "n_estimators": hp.quniform("n_estimators", 200, 2500, 50),
            "max_depth": hp.quniform("max_depth", 3, 12, 1),
            "min_child_weight": hp.loguniform("min_child_weight", np.log(0.1), np.log(10)),
            "subsample": hp.uniform("subsample", 0.6, 1.0),
            "colsample_bytree": hp.uniform("colsample_bytree", 0.6, 1.0),
            "gamma": hp.loguniform("gamma", np.log(1e-8), np.log(5.0)),
            "reg_alpha": hp.loguniform("reg_alpha", np.log(1e-8), np.log(10.0)),
            "reg_lambda": hp.loguniform("reg_lambda", np.log(1e-8), np.log(10.0)),
            "objective": hp.choice("objective", ["reg:squarederror", "reg:pseudohubererror"]),
            "eval_metric": hp.choice("eval_metric", ["rmse", "mae"]),
            "random_state": 42,
            "n_jobs": -1,
            "verbosity": 0}

In [ ]:
def hyperopt_rfr_low(params, cv=3, X=X_train_low, y=y_train_low, random_state=rs):
    params["n_estimators"] = int(params["n_estimators"])
    params["max_depth"]    = int(params["max_depth"])

    model = XGBRegressor(**params)
    
    scores = cross_val_score(model, X, y, cv=cv, scoring='neg_mean_squared_error', n_jobs=-1)
    mean_score = scores.mean()
    # --- возврат результата ------------------------------------------------
    return {'loss': -mean_score,
            'status': STATUS_OK,
            'model': model,
            'params': params}

In [ ]:
# Подбор гиперпараметров
trials_low = Trials() # используется для логирования результатов

if recalculating_model_low:
    best_down=fmin(hyperopt_rfr_low, # функция 
                   space=space_low, # пространство гиперпараметров
                   algo=tpe.suggest, # алгоритм оптимизации, установлен по умолчанию, задавать необязательно
                   max_evals=15, # максимальное количество итераций
                   trials=trials_low, # логирование результатов
                   rstate=np.random.default_rng(rs))# фиксируем для повторяемости результата

In [ ]:
# Вывод результатов подбора
if recalculating_model_low:
    best_trial_low = trials_low.best_trial
    model_low_hyp_res = best_trial_low['result']['model']
    print("\n=== Параметры ===")
    print(best_trial_low['result']['params'])
    print("\n=== Средний CV-score (MSE) ===")
    print(-best_trial_low['result']['loss'])
else:
    params_low = {}
    print('Параметры модели не пересчитывались\nСредний CV-score (MSE) 0.025091218820831842')

Подбор гиперпараметров осуществлен при пороге для определения направления движения цены в процентах
level_price_movement_direction = 0.15:  
100%|██████████| 7/7 [5:55:43<00:00, 3049.07s/trial, best loss: 0.13640509813247906]  
Наилучшие значения гиперпараметров {'bootstrap': False, 'max_depth': 16.0, 'max_features': 0.7562196798242258, 'min_samples_leaf': 9.0, 'model_type': 'rf', 'n_estimators': 330.0}  
Для перезапуска подбора гиперпараметров изменить recalculating_model_low на True.

100%|██████████| 15/15 [13:21:44<00:00, 3206.95s/trial, best loss: 0.025512210389043916]
{'bootstrap': False, 'max_depth': 20.0, 'max_features': 0.7014226785922257, 'min_samples_leaf': 1.0, 'model_type': 'rf', 'n_estimators': 400.0}

100%|██████████| 15/15 [1:34:50<00:00, 379.39s/trial, best loss: 0.0032014550330738225]
=== Параметры ===
{'colsample_bytree': 0.8422597946469974, 'eval_metric': 'mae', 'gamma': 0.00026731209746217214, 'learning_rate': 0.032452172710781914, 'max_depth': 11, 'min_child_weight': 1.2743849764010975, 'n_estimators': 2400, 'n_jobs': -1, 'objective': 'reg:squarederror', 'random_state': 42, 'reg_alpha': 2.5996973016007112e-06, 'reg_lambda': 2.3514904339337348e-08, 'subsample': 0.6138813743427279, 'verbosity': 0}
=== Средний CV-score (MSE) ===
-0.0032014550330738225

In [ ]:
# Обучение модели и получение ее оценки
if not recalculating_model_low:
    model_low_hyp_res = GradientBoostingRegressor(**params_low)
model_low_hyp_res.fit(X_train_low, y_train_low)
y_pred_train_low = model_low_hyp_res.predict(X_train_low)
y_pred_test_low = model_low_hyp_res.predict(X_test_low)
# Получение метрик модели
print('-------На тренировочных данных-------')
print(f'MAE: {metrics.mean_absolute_error(y_train_low, y_pred_train_low):.4f}    MSE: {metrics.mean_squared_error(y_train_low, y_pred_train_low):.4f}')
print('-------На тестовых данных------------')
print(f'MAE: {metrics.mean_absolute_error(y_test_low, y_pred_test_low):.4f}    MSE: {metrics.mean_squared_error(y_test_low, y_pred_test_low):.4f}')

Полученные модели показывают удовлетворительные результаты по оценочным метрикам. Поэтому обучаем полученные модели на полных данных и сохраняем в файл для дальнейшего построения торгово-рекомендательной системы.

## Обучение моделей на полных данных и сохранение в файл

In [ ]:
# Обучение моделей на полных данных
model_to_up_hyp_res.fit(X, y_to_up)
model_to_down_hyp_res.fit(X, y_to_down)
model_high_hyp_res.fit(X, y_high)
model_low_hyp_res.fit(X, y_low)
model_level_hyp_res.fit(X, y_level)

# Сохранение моделей в файл
joblib.dump(model_to_up_hyp_res, './models/model_to_up.pkl')
joblib.dump(model_to_down_hyp_res, './models/model_to_down.pkl')
joblib.dump(model_high_hyp_res, './models/model_high.pkl')
joblib.dump(model_low_hyp_res, './models/model_low.pkl')
joblib.dump(model_level_hyp_res, './models/model_level.pkl')

---

> При соединении с новостями сохранять новость в свечах до 90 минут, хранить 2 последние новости + 2 последних отчета, новость сохранять в течении часа, отчеты - дольше, а лучше ввести признак "минут с момента публикации новости/отчета"

> Можно добавить признак работы американской биржи/бирж других стран - True/False